# Interim five-model comparison: DeepJSCC, ADJSCC, NTSCC_plus, and SwinJSCC

This Kaggle notebook assembles the **completed** AWGN and Rayleigh training runs, including the paper-based DeepJSCC-Q adaptation and their Kodak results. Run All regenerates the comparison tables, plots, and downloadable report CSVs from embedded copies of recorded outputs. It does not retrain the networks; each training run is linked below.

All three models completed 150 epochs per channel at 10 dB on DIV2K and were evaluated on all 24 Kodak images. DeepJSCC and ADJSCC each sampled 800 DIV2K images per epoch (128px crops). NTSCC_plus sampled 128 distinct images per epoch (256px crops), so its 150 epochs were not 150 full DIV2K passes.

The experiment named NTSCC++ uses the released author-linked **NTSCC_plus** implementation; it is not the unreleased online-editing NTSCC++ implementation. Rayleigh support for NTSCC_plus and ADJSCC is an extension of their author-release AWGN pipeline.

## Original training runs

- [DeepJSCC — 150 epochs/channel, T4 ×2](https://www.kaggle.com/code/hmmkk20/deepjscc-div2k-kodak-150ep)
- [ADJSCC — 150 epochs/channel](https://www.kaggle.com/code/hmmkk20/adjscc-div2k-kodak-150ep-20261016)
- [NTSCC_plus — 150 epochs/channel, T4 ×2](https://www.kaggle.com/code/hmmkk20/ntscc-plus-div2k-kodak-150ep-20261016)
- [DeepJSCC FLOP profiler](https://www.kaggle.com/code/hmmkk20/deepjscc-flops-profile-20261017)

## Interpretation limits

- The analog models use nominal CBR 1/6; NTSCC_plus uses about 0.047–0.049. Quality is not rate-matched.
- FLOP counts are profiler estimates. Profile sizes differ (128px crop, 256px crop, or full Kodak image), optimizer updates are excluded, and fused operators may be under-counted.
- This notebook embeds summary and detailed metrics for all runs, full available DeepJSCC and NTSCC_plus epoch histories, and available per-image Kodak rows. ADJSCC per-image and epoch CSVs remain visible in its linked Kaggle output.


This is an interim snapshot: SQ-GAN, DDM-SSCC, DiT-JSCC, and DeepJSCC-Q are not yet included because their matching runs are not complete.

**Rate and compute comparability:** these runs do not share one channel-use rate. DeepJSCC/ADJSCC use CBR 1/6, NTSCC_plus is about 0.047–0.049, and SwinJSCC is 0.0625. Treat quality values as measured operating points, not a fixed-rate ranking. FLOPs use each model's listed profile dimensions and operator coverage.

In [ ]:
import io, json, math, warnings
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Embedded artifacts are snapshots downloaded from the completed Kaggle runs.
SUMMARY_CSV = 'model,channel,snr_db,epochs,kodak_psnr_db,kodak_ssim,kodak_msssim,mean_complex_channel_uses_per_image,mean_cbr\r\nDeepJSCC,AWGN,10.0,150,23.368306047249074,0.6450856924057007,0.8727623720963796,196608.0,0.1666666666666666\r\nDeepJSCC,Rayleigh,10.0,150,20.33186242830112,0.5323714216550192,0.7446102251609167,196608.0,0.1666666666666666\r\nADJSCC,AWGN,10.0,150,32.225783348083496,0.9200940628846488,0.9748047590255736,196608.0,0.1666666666666666\r\nADJSCC,Rayleigh,10.0,150,29.80873688062032,0.8685299406449,0.9542806545893352,196608.0,0.1666666666666666\r\nNTSCC_plus,AWGN,10.0,150,21.12683176994324,0.5454493500292301,0.7897162660956383,55442.333333333336,0.0469990483036747\r\nNTSCC_plus,Rayleigh,10.0,150,21.00322651863098,0.5700619394580523,0.7956210548679034,57522.333333333336,0.0487622861508969\r\nSwinJSCC,AWGN,10.0,150,28.98231709285191,0.8347020546595255,0.9572386716802916,73728.0,0.0625\r\nSwinJSCC,Rayleigh,10.0,150,26.834627616858462,0.7676363637049993,0.931297724445661,73728.0,0.0625\r\nDeepJSCC-Q (paper-based),AWGN,10.0,150,23.510403119588915,0.6365299113094807,0.8507442300518354,196608.0,0.1666666666666666\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,150,22.9092481429779,0.5787361661593119,0.7958295680582523,196608.0,0.1666666666666666\r\n'
DETAIL_CSV = 'model,channel,snr_db,epochs,kodak_psnr_db,kodak_ssim,kodak_msssim,mean_complex_channel_uses_per_image,mean_cbr,div2k_pool_images,div2k_images_sampled_per_epoch,crop_px,batch_size,mixed_precision,training_flops_per_sample,training_flops_profile_input,inference_flops_per_image,inference_flops_profile_input,kaggle_kernel\r\nDeepJSCC,AWGN,10.0,150,23.368306047249074,0.6450856924057007,0.8727623720963796,196608.0,0.1666666666666666,800,800,128,32,false,167346181,one 128x128 RGB sample (forward + backward),167182338,one 128x128 RGB sample,https://www.kaggle.com/code/hmmkk20/deepjscc-div2k-kodak-150ep\r\nDeepJSCC,Rayleigh,10.0,150,20.33186242830112,0.5323714216550192,0.7446102251609167,196608.0,0.1666666666666666,800,800,128,32,false,167362567,one 128x128 RGB sample (forward + backward),167182340,one 128x128 RGB sample,https://www.kaggle.com/code/hmmkk20/deepjscc-div2k-kodak-150ep\r\nADJSCC,AWGN,10.0,150,32.225783348083496,0.9200940628846488,0.9748047590255736,196608.0,0.1666666666666666,800,800,128,16,false,70469104100,one 128x128 RGB sample (forward + backward; Adam update excluded),23490280974,one 128x128 RGB sample,https://www.kaggle.com/code/hmmkk20/adjscc-div2k-kodak-150ep-20261016\r\nADJSCC,Rayleigh,10.0,150,29.80873688062032,0.8685299406449,0.9542806545893352,196608.0,0.1666666666666666,800,800,128,16,false,70469112292,one 128x128 RGB sample (forward + backward; Adam update excluded),23490289166,one 128x128 RGB sample,https://www.kaggle.com/code/hmmkk20/adjscc-div2k-kodak-150ep-20261016\r\nNTSCC++ (NTSCC_plus release),AWGN,10.0,150,21.12683176994324,0.5454493500292301,0.7897162660956383,55442.333333333336,0.0469990483036747,800,128,256,8,true (FP16 autocast),225518008470,one 256x256 RGB crop (PyTorch profiler; fused NATTEN may be under-counted),481300316170,one full-resolution 768x512 Kodak image (PyTorch profiler; fused NATTEN may be under-counted),https://www.kaggle.com/code/hmmkk20/ntscc-plus-div2k-kodak-150ep-20261016\r\nNTSCC++ (NTSCC_plus release),Rayleigh,10.0,150,21.00322651863098,0.5700619394580523,0.7956210548679034,57522.333333333336,0.0487622861508969,800,128,256,8,true (FP16 autocast),225518090394,one 256x256 RGB crop (PyTorch profiler; fused NATTEN may be under-counted),481300316174,one full-resolution 768x512 Kodak image (PyTorch profiler; fused NATTEN may be under-counted),https://www.kaggle.com/code/hmmkk20/ntscc-plus-div2k-kodak-150ep-20261016\r\nSwinJSCC,AWGN,10.0,150,28.98231709285191,0.8347020546595255,0.9572386716802916,73728.0,0.0625,800,800,256,16,true (FP16 autocast),206543376388,one 256x256 RGB sample (forward + backward; PyTorch profiler),68925788161,one 256x256 RGB image (PyTorch profiler),https://www.kaggle.com/code/hmmkk20/swinjscc-div2k-kodak-150ep-20261018\r\nSwinJSCC,Rayleigh,10.0,150,26.834627616858462,0.7676363637049993,0.931297724445661,73728.0,0.0625,800,800,256,16,true (FP16 autocast),206543413252,one 256x256 RGB sample (forward + backward; PyTorch profiler),68925788161,one 256x256 RGB image (PyTorch profiler),https://www.kaggle.com/code/hmmkk20/swinjscc-div2k-kodak-150ep-20261018\r\nDeepJSCC-Q (paper-based),AWGN,10.0,150,23.510403119588915,0.6365299113094807,0.8507442300518354,196608.0,0.1666666666666666,800,800,128,32,false,168034406,one 128x128 RGB sample (forward + backward),167878658,one 128x128 RGB sample,https://www.kaggle.com/code/hmmkk20/deepjscc-q-div2k-kodak-150ep-20261018\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,150,22.9092481429779,0.5787361661593119,0.7958295680582523,196608.0,0.1666666666666666,800,800,128,32,false,168116335,one 128x128 RGB sample (forward + backward),167895051,one 128x128 RGB sample,https://www.kaggle.com/code/hmmkk20/deepjscc-q-div2k-kodak-150ep-20261018\r\n'
summary = pd.read_csv(io.StringIO(SUMMARY_CSV))
detail = pd.read_csv(io.StringIO(DETAIL_CSV))
assert len(summary) == 10 and set(summary.epochs.astype(int)) == {150}
assert set(summary.channel) == {"AWGN", "Rayleigh"}
display(Markdown("## Completed runs: 150 epochs per model and channel; 24 Kodak images per run"))
display(summary)


In [ ]:
# Quality metrics. Channel-use rates differ by model; this is not a fixed-rate ranking.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, metric, title in zip(axes, ["kodak_psnr_db", "kodak_ssim", "kodak_msssim"], ["PSNR (dB)", "SSIM", "MS-SSIM"]):
    pivot = summary.pivot(index="model", columns="channel", values=metric)
    pivot.plot(kind="bar", ax=ax, rot=18, color=["#377eb8", "#e41a1c"])
    ax.set_title(title); ax.set_xlabel(""); ax.grid(axis="y", alpha=.25); ax.legend(title="Channel")
fig.tight_layout(); plt.show()

display(Markdown("## Detailed compute and channel accounting"))
display(detail[["model","channel","epochs","training_flops_per_sample","training_flops_profile_input","inference_flops_per_image","inference_flops_profile_input","mean_complex_channel_uses_per_image","mean_cbr"]])


In [ ]:
# FLOPs are profiler estimates. Profile dimensions and operator coverage differ by implementation.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric, title in zip(axes, ["training_flops_per_sample", "inference_flops_per_image"], ["Training FLOPs/sample", "Inference FLOPs/image"]):
    vals = detail.copy(); vals[metric] = pd.to_numeric(vals[metric], errors="coerce")
    vals["series"] = vals["model"] + " / " + vals["channel"]
    ax.barh(vals["series"], vals[metric], color=["#4daf4a" if c=="AWGN" else "#984ea3" for c in vals.channel])
    ax.set_xscale("log"); ax.set_title(title); ax.grid(axis="x", alpha=.25)
fig.tight_layout(); plt.show()


In [ ]:
# Epoch histories: full CSV records are embedded for DeepJSCC and NTSCC_plus.
EPOCH_CSVS = {'DeepJSCC_AWGN': 'model,channel,snr_db,epoch,train_mse,epoch_seconds,c,nominal_cbr\nDeepJSCC,AWGN,10.0,1,0.09982317596673966,4.197293519973755,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,2,0.06375875070691109,1.2483015060424805,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,3,0.03912618324160576,1.2486529350280762,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,4,0.03003191575407982,1.2438199520111084,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,5,0.025663421526551246,1.2595458030700684,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,6,0.02328482925891876,1.3676705360412598,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,7,0.021647571623325347,1.240717887878418,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,8,0.019484945088624955,1.241450309753418,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,9,0.018165112882852556,1.2474720478057861,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,10,0.01666639231145382,1.2651138305664062,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,11,0.016413725800812243,1.294379472732544,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,12,0.016003673635423182,1.2308876514434814,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,13,0.015007449239492416,1.2432050704956055,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,14,0.0145833295956254,1.38509202003479,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,15,0.01415074646472931,1.2371642589569092,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,16,0.01343945525586605,1.2746000289916992,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,17,0.01287841983139515,1.254958152770996,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,18,0.013399349227547645,1.2428641319274902,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,19,0.012495021298527718,1.2868523597717285,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,20,0.012619686163961887,1.2870957851409912,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,21,0.012144133262336255,1.2532305717468262,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,22,0.011913248337805271,1.39028000831604,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,23,0.011998238861560822,1.2947683334350586,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,24,0.011639730967581273,1.2684316635131836,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,25,0.011416011452674867,1.2873611450195312,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,26,0.011500311754643918,1.2781400680541992,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,27,0.011475364379584789,1.2463982105255127,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,28,0.01107743714004755,1.267517328262329,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,29,0.010988340433686971,1.3394780158996582,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,30,0.011021783500909805,1.3100380897521973,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,31,0.01080432053655386,1.278719186782837,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,32,0.010530222337692977,1.266674518585205,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,33,0.010336952023208142,1.2644257545471191,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,34,0.010328033044934272,1.3265788555145264,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,35,0.00999542374163866,1.249962329864502,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,36,0.010263652093708515,1.3006598949432373,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,37,0.010204471498727798,1.399427890777588,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,38,0.010429442543536425,1.3479549884796143,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,39,0.010216240808367728,1.286186933517456,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,40,0.010018894560635089,1.2691676616668701,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,41,0.009977224431931973,1.2775771617889404,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,42,0.009743478372693062,1.2893404960632324,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,43,0.009475208856165408,1.3251430988311768,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,44,0.009837986193597317,1.2866928577423096,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,45,0.00983631808310747,1.4124252796173096,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,46,0.009765866100788117,1.2834973335266113,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,47,0.00961010968312621,1.2574851512908936,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,48,0.009458941631019116,1.2946953773498535,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,49,0.009197963401675224,1.2906019687652588,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,50,0.009151438660919666,1.3117551803588867,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,51,0.008925956636667252,1.303335428237915,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,52,0.009150139093399047,1.3583059310913086,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,53,0.009193448033183813,1.2942049503326416,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,54,0.008770993575453759,1.278254508972168,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,55,0.00892398588359356,1.2999489307403564,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,56,0.009070281684398652,1.3069357872009277,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,57,0.008906671702861786,1.3163034915924072,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,58,0.00899079605937004,1.3628363609313965,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,59,0.008858937583863736,1.3623781204223633,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,60,0.00893871819600463,1.4616565704345703,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,61,0.008423540126532316,1.379606008529663,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,62,0.008555956911295652,1.39613938331604,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,63,0.008716745972633362,1.2985515594482422,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,64,0.008741822596639394,1.2910737991333008,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,65,0.008493680190294981,1.2930712699890137,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,66,0.008196206167340278,1.341245174407959,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,67,0.008119294438511133,1.4606339931488037,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,68,0.008359766229987145,1.3642446994781494,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,69,0.00828644772991538,1.34419584274292,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,70,0.008153857663273811,1.3322155475616455,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,71,0.008418982066214085,1.376147747039795,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,72,0.008263944666832685,1.3516874313354492,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,73,0.008471771869808435,1.38144850730896,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,74,0.008381890449672937,1.4625647068023682,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,75,0.008253294564783574,1.348433017730713,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,76,0.008343711905181407,1.3706138134002686,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,77,0.008208495862782002,1.3479597568511963,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,78,0.008444862477481364,1.3250653743743896,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,79,0.008127294853329659,1.363865852355957,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,80,0.007984977215528488,1.3521678447723389,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,81,0.008077950403094291,1.303823471069336,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,82,0.008186741042882204,1.3327536582946777,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,83,0.007946942094713449,1.3560853004455566,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,84,0.008383473586291075,1.3006889820098877,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,85,0.007934900652617215,1.3246047496795654,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,86,0.007949960548430682,1.3264949321746826,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,87,0.007977758925408125,1.3676364421844482,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,88,0.007794033382087946,1.4231922626495361,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,89,0.007707967925816774,1.4578447341918945,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,90,0.007712960634380579,1.4222297668457031,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,91,0.007955238055437804,1.4215939044952393,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,92,0.00775817833840847,1.3582863807678223,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,93,0.00801266822963953,1.3873860836029053,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,94,0.007615770921111107,1.3073632717132568,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,95,0.007753511071205139,1.3211603164672852,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,96,0.007864887956529856,1.4439818859100342,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,97,0.007600155100226403,1.3596558570861816,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,98,0.007594758570194244,1.3473353385925293,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,99,0.007706051636487246,1.3433876037597656,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,100,0.007553027756512165,1.3383145332336426,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,101,0.007854789607226848,1.417949914932251,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,102,0.0077864166162908074,1.3859453201293945,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,103,0.007718152906745672,1.4340848922729492,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,104,0.007514148093760014,1.3446428775787354,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,105,0.007404153030365706,1.3738934993743896,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,106,0.007357468791306019,1.334542989730835,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,107,0.007695039100944996,1.3988866806030273,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,108,0.007577766571193934,1.3338558673858643,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,109,0.007402731068432331,1.394545078277588,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,110,0.007264422252774239,1.3977515697479248,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,111,0.00744859829545021,1.468252420425415,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,112,0.007510757632553577,1.3463153839111328,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,113,0.007501307874917984,1.319075107574463,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,114,0.007319371812045574,1.3614006042480469,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,115,0.007479765210300684,1.3629536628723145,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,116,0.007761803381145,1.412583827972412,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,117,0.007343068718910217,1.363081693649292,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,118,0.007775384988635778,1.439662218093872,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,119,0.007296020314097405,1.3627588748931885,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,120,0.007647427096962929,1.4137251377105713,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,121,0.007414277959614992,1.3958065509796143,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,122,0.007397226002067328,1.3795077800750732,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,123,0.007430732827633619,1.345978021621704,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,124,0.007332437355071306,1.3604121208190918,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,125,0.0074461490660905835,1.481565237045288,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,126,0.007402000464498997,1.4142119884490967,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,127,0.00720812726765871,1.4079816341400146,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,128,0.007239011172205209,1.397003173828125,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,129,0.0070599007420241834,1.3957281112670898,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,130,0.0074168663285672665,1.4378268718719482,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,131,0.007290898989886045,1.404306411743164,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,132,0.00702288968488574,1.4992880821228027,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,133,0.00710004085674882,1.4262778759002686,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,134,0.006908147595822811,1.4234752655029297,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,135,0.00742160951718688,1.3998866081237793,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,136,0.007000453043729067,1.4671103954315186,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,137,0.007239385563880205,1.4343905448913574,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,138,0.0073544041812419894,1.451740026473999,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,139,0.007178279664367437,1.4962406158447266,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,140,0.0073311725258827206,1.4645228385925293,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,141,0.007100623659789562,1.4393725395202637,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,142,0.007237421479076147,1.4053130149841309,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,143,0.007006293628364801,1.419389247894287,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,144,0.007101038731634617,1.4187078475952148,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,145,0.007202534228563309,1.4441478252410889,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,146,0.00709148844704032,1.5003104209899902,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,147,0.00712133502587676,1.3766586780548096,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,148,0.00727725125849247,1.435020923614502,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,149,0.007260240036994219,1.464857816696167,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,150,0.0072437833994627,1.446413278579712,8,0.16666666666666666\n', 'DeepJSCC_Rayleigh': 'model,channel,snr_db,epoch,train_mse,epoch_seconds,c,nominal_cbr\nDeepJSCC,Rayleigh,10.0,1,0.0988293132185936,1.597193717956543,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,2,0.0882253736257553,1.463083028793335,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,3,0.07116201475262642,1.4578368663787842,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,4,0.05853131726384163,1.4535129070281982,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,5,0.05075315684080124,1.445157766342163,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,6,0.04858627155423165,1.457226037979126,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,7,0.04673305854201317,1.4136710166931152,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,8,0.044232844933867456,1.5906472206115723,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,9,0.03865815915167332,1.4250385761260986,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,10,0.036325090304017066,1.4399840831756592,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,11,0.038596205040812494,1.4724342823028564,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,12,0.03582734256982803,1.4206819534301758,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,13,0.03718619398772716,1.462951421737671,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,14,0.03512484960258007,1.4924190044403076,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,15,0.03785569511353969,1.5184516906738281,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,16,0.03502297028899193,1.4803264141082764,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,17,0.03368144869804382,1.423351526260376,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,18,0.03386340901255608,1.4553024768829346,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,19,0.033019354790449144,1.4871439933776855,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,20,0.03300804637372494,1.4415359497070312,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,21,0.03335367746651172,1.5045230388641357,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,22,0.03202147789299488,1.4674978256225586,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,23,0.03484632126986981,1.4657583236694336,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,24,0.03417222328484058,1.4673707485198975,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,25,0.03414089776575566,1.490532636642456,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,26,0.030976130366325377,1.4845831394195557,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,27,0.028612087368965148,1.4844019412994385,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,28,0.02981865018606186,1.5844879150390625,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,29,0.03191207237541675,1.5371172428131104,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,30,0.030501200780272485,1.4770023822784424,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,31,0.029671077355742453,1.4826693534851074,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,32,0.02943895883858204,1.4557816982269287,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,33,0.026856513023376466,1.4267847537994385,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,34,0.027480105832219125,1.5391700267791748,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,35,0.02854961596429348,1.5536205768585205,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,36,0.029531653448939325,1.4514806270599365,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,37,0.027866246551275252,1.4345736503601074,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,38,0.02757535398006439,1.4752345085144043,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,39,0.027908382937312126,1.4671821594238281,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,40,0.027402280643582345,1.4954724311828613,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,41,0.028959313929080962,1.5561540126800537,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,42,0.026062045767903327,1.5009219646453857,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,43,0.025406382977962494,1.513467788696289,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,44,0.025177910067141054,1.49113130569458,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,45,0.029293771535158157,1.465512752532959,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,46,0.029257982671260833,1.5031514167785645,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,47,0.03038719519972801,1.510906457901001,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,48,0.027292762771248816,1.5649425983428955,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,49,0.026416086554527284,1.5262961387634277,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,50,0.02573420315980911,1.4904747009277344,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,51,0.02554155685007572,1.5234034061431885,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,52,0.024267013594508172,1.489872932434082,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,53,0.02627017389982939,1.4571149349212646,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,54,0.025388200357556342,1.5193336009979248,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,55,0.0251073032617569,1.530482530593872,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,56,0.025182886719703673,1.5103492736816406,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,57,0.025039794966578485,1.4844841957092285,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,58,0.02447968170046806,1.51027512550354,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,59,0.023596500977873802,1.496868371963501,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,60,0.0226706163585186,1.591428518295288,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,61,0.023505417928099633,1.5771687030792236,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,62,0.022282766923308373,1.5081360340118408,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,63,0.02407857611775398,1.5308072566986084,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,64,0.02591887779533863,1.5710716247558594,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,65,0.026039888933300972,1.5558679103851318,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,66,0.02432307682931423,1.5216679573059082,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,67,0.022614075914025308,1.5368421077728271,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,68,0.02234252654016018,1.5855157375335693,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,69,0.022976682856678962,1.5497357845306396,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,70,0.02195304997265339,1.5486462116241455,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,71,0.022741339877247812,1.4931304454803467,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,72,0.02258528456091881,1.5485758781433105,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,73,0.022945038266479967,1.5374603271484375,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,74,0.02365014549344778,1.5844941139221191,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,75,0.022869915515184403,1.5373375415802002,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,76,0.02326191745698452,1.5367207527160645,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,77,0.023500179499387742,1.5474178791046143,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,78,0.022383475229144096,1.5625922679901123,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,79,0.020673485547304152,1.5725734233856201,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,80,0.022309423461556434,1.5702595710754395,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,81,0.022667487934231758,1.6005971431732178,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,82,0.02201342597603798,1.6071679592132568,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,83,0.022574724406003954,1.5695550441741943,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,84,0.021212293952703475,1.56129789352417,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,85,0.021025249734520912,1.5727767944335938,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,86,0.021871483102440834,1.5381357669830322,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,87,0.02148771770298481,1.5715935230255127,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,88,0.023758746571838855,1.5584802627563477,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,89,0.021549981124699115,1.5860183238983154,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,90,0.01865873608738184,1.5749905109405518,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,91,0.020049177408218384,1.5336189270019531,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,92,0.020623030215501784,1.5211122035980225,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,93,0.02077210061252117,1.571976900100708,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,94,0.020936616510152817,1.5796802043914795,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,95,0.022118094302713873,1.5299005508422852,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,96,0.020740063451230525,1.5830152034759521,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,97,0.01954802133142948,1.5650575160980225,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,98,0.018753526844084264,1.5382802486419678,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,99,0.01928467594087124,1.61635422706604,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,100,0.020221721716225147,1.5727787017822266,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,101,0.01929360080510378,1.557955026626587,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,102,0.019791422635316847,1.5425701141357422,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,103,0.018006792366504668,1.5355961322784424,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,104,0.01752388771623373,1.6423139572143555,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,105,0.01803989376872778,1.5988991260528564,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,106,0.019292038790881635,1.6636972427368164,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,107,0.020081962868571283,1.5554664134979248,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,108,0.020052433125674726,1.5534868240356445,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,109,0.019953731037676333,1.603933572769165,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,110,0.019694149680435657,1.5636703968048096,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,111,0.019494151435792445,1.5822851657867432,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,112,0.02060811948031187,1.6193454265594482,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,113,0.019015569649636747,1.5896899700164795,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,114,0.01655249897390604,1.5861849784851074,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,115,0.018185013458132743,1.5972323417663574,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,116,0.0184220689535141,1.5713706016540527,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,117,0.0168682262301445,1.567946195602417,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,118,0.016526799872517584,1.6819300651550293,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,119,0.018931348323822022,1.6390933990478516,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,120,0.018434323370456696,1.6078271865844727,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,121,0.016666100434958933,1.5605478286743164,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,122,0.015932389162480833,1.5637314319610596,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,123,0.016874795742332936,1.5854747295379639,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,124,0.015832162387669088,1.6116700172424316,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,125,0.016968960165977477,1.6217210292816162,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,126,0.016113923005759714,1.6291730403900146,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,127,0.016370865143835546,1.5779430866241455,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,128,0.017340524382889272,1.5891809463500977,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,129,0.018038660734891892,1.573983907699585,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,130,0.017166455984115602,1.6318843364715576,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,131,0.016507445611059667,1.5992960929870605,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,132,0.015154939368367195,1.5834152698516846,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,133,0.015313753113150596,1.6159775257110596,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,134,0.015508278720080853,1.604085922241211,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,135,0.01628334168344736,1.572537899017334,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,136,0.01723056297749281,1.636871099472046,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,137,0.016221270859241486,1.643869161605835,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,138,0.01639403559267521,1.6180667877197266,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,139,0.016439789570868016,1.5545861721038818,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,140,0.01745746858417988,1.6156814098358154,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,141,0.014962810017168522,1.6638095378875732,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,142,0.015858321227133276,1.6097309589385986,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,143,0.017720992118120192,1.6446242332458496,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,144,0.017697106152772903,1.6394307613372803,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,145,0.017165479250252247,1.6477539539337158,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,146,0.015033623985946178,1.6500158309936523,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,147,0.013815610259771348,1.6373653411865234,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,148,0.01356470439583063,1.5946767330169678,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,149,0.015874837674200535,1.630296230316162,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,150,0.015283494219183923,1.6405420303344727,8,0.16666666666666666\n', 'NTSCC_plus_AWGN': 'model,channel,epoch,epochs_target,snr_db,train_loss,train_psnr_db,estimated_bpp,epoch_seconds,train_flops_per_step,inference_flops_per_image,batch_size,crop,images_in_epoch,amp_fp16,source_commit\nNTSCC_plus,AWGN,1,150,10.0,39792.952880859375,-2.755332043364393,12.56716638803482,31.988705158233643,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,2,150,10.0,6981.579391479492,5.323929601522509,5.221501022577286,27.936014413833618,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,3,150,10.0,2899.8765716552734,9.230145971967366,2.0459309965372086,22.147674798965454,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,4,150,10.0,2035.117774963379,10.467426791402856,1.4391547292470932,21.931376695632935,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,5,150,10.0,1534.4702911376953,11.601195251592436,1.3555161133408546,22.36415433883667,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,6,150,10.0,1367.8081321716309,11.676967175571765,1.3413746878504753,21.816887855529785,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,7,150,10.0,1034.2150039672852,12.988741757761888,1.4124318361282349,22.35653328895569,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,8,150,10.0,860.473949432373,13.691158757316932,1.3835690394043922,21.719832181930542,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,9,150,10.0,790.1523857116699,14.119188622652885,1.403208665549755,21.811223030090332,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,10,150,10.0,753.8871574401855,14.208758550937098,1.3850225806236267,22.316543579101562,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,11,150,10.0,592.3934116363525,15.373168726706194,1.3910673558712006,21.595791578292847,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,12,150,10.0,594.8851890563965,15.277154761942482,1.296489842236042,21.9667387008667,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,13,150,10.0,531.2410869598389,15.730459440883573,1.2792765721678734,21.560975074768066,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,14,150,10.0,542.2916355133057,15.63637102826535,1.2845947816967964,22.095397233963013,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,15,150,10.0,451.8876552581787,16.46690545853577,1.2610168308019638,21.74895143508911,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,16,150,10.0,491.3628578186035,15.877132800143066,1.2510557249188423,21.912362813949585,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,17,150,10.0,428.53265953063965,16.47285263476945,1.2621963173151016,23.17900586128235,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,18,150,10.0,442.77608489990234,16.334588785847227,1.290684811770916,22.55223536491394,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,19,150,10.0,430.241662979126,16.399056273208718,1.2986488118767738,23.259185552597046,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,20,150,10.0,462.53583908081055,15.939231115566113,1.349912278354168,22.78720736503601,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,21,150,10.0,409.3742847442627,16.54609024445658,1.4328963607549667,23.259812355041504,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,22,150,10.0,375.34513092041016,16.84652980449987,1.3376842513680458,22.74508547782898,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,23,150,10.0,336.80224800109863,17.28211327336637,1.31114911288023,23.555722951889038,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,24,150,10.0,353.61078453063965,17.003364493072716,1.321211315691471,23.02208423614502,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,25,150,10.0,365.6126117706299,16.81016237398682,1.3191985934972763,22.953298568725586,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,26,150,10.0,351.7023410797119,17.02427352948228,1.307821087539196,22.95594096183777,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,27,150,10.0,333.9455795288086,17.248168283985436,1.32345799356699,22.598791122436523,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,28,150,10.0,289.34821128845215,17.791647165157254,1.3124936148524284,23.652023792266846,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,29,150,10.0,350.45277309417725,17.004256318685318,1.3712836056947708,22.952850580215454,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,30,150,10.0,323.0689353942871,17.39843962710249,1.3645784258842468,22.527425527572632,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,31,150,10.0,332.90099239349365,17.276934549151317,1.3872633501887321,22.806499481201172,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,32,150,10.0,306.04050731658936,17.607156134288516,1.3718416839838028,22.44851326942444,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,33,150,10.0,267.94407749176025,18.145132397726066,1.3385836035013199,22.946462631225586,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,34,150,10.0,279.67390727996826,17.872319494093077,1.3454597815871239,22.650651693344116,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,35,150,10.0,298.9259195327759,17.632541580221503,1.352270171046257,22.443472623825073,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,36,150,10.0,288.36194896698,17.77113654188968,1.4059413000941277,22.69043779373169,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,37,150,10.0,275.2576484680176,18.00000496322244,1.3879760652780533,22.86424708366394,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,38,150,10.0,286.4496536254883,17.79766778517618,1.4043457582592964,23.44990634918213,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,39,150,10.0,257.75805854797363,18.27785808580352,1.3741118535399437,22.95795178413391,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,40,150,10.0,272.07842922210693,18.044610782181564,1.387479528784752,24.07218647003174,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,41,150,10.0,310.02437591552734,17.42095518306415,1.488906778395176,23.068941593170166,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,42,150,10.0,281.722225189209,17.81035442697867,1.4568607732653618,22.8352530002594,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,43,150,10.0,287.3939914703369,17.760138114826677,1.5205247327685356,23.265828371047974,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,44,150,10.0,257.0302438735962,18.23690604931171,1.4466557875275612,22.829960346221924,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,45,150,10.0,252.5364475250244,18.29284502596057,1.4601499736309052,23.61336851119995,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,46,150,10.0,249.19870567321777,18.441641474107072,1.4597879201173782,23.415640592575073,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,47,150,10.0,250.6742811203003,18.394340787942603,1.4825955629348755,23.85462760925293,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,48,150,10.0,219.5273265838623,18.95227050864638,1.4203071519732475,22.772010564804077,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,49,150,10.0,218.76111507415771,18.972722256513133,1.426989734172821,23.55838966369629,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,50,150,10.0,214.8576898574829,19.049966408820033,1.4618085250258446,22.798104524612427,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,51,150,10.0,218.01782512664795,18.980944165588458,1.4519735351204872,22.424926280975342,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,52,150,10.0,263.0146884918213,18.11001973380931,1.5207263007760048,22.731394290924072,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,53,150,10.0,238.76737117767334,18.528033344456365,1.5283488929271698,22.195539236068726,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,54,150,10.0,222.07004261016846,18.9008735595314,1.5059731230139732,22.692216396331787,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,55,150,10.0,221.9753828048706,18.913789704016907,1.468542106449604,22.078246593475342,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,56,150,10.0,201.6249485015869,19.322658963723264,1.4527445286512375,22.057023286819458,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,57,150,10.0,233.31722450256348,18.67228869039944,1.4722024202346802,22.71575093269348,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,58,150,10.0,231.58963108062744,18.723767035119167,1.501148760318756,22.49824619293213,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,59,150,10.0,217.8218593597412,18.96824797260305,1.488989733159542,22.603149890899658,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,60,150,10.0,223.33054065704346,18.835760042640356,1.498335748910904,22.036519527435303,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,61,150,10.0,210.6287441253662,19.099680811280052,1.491656482219696,22.118987798690796,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,62,150,10.0,215.98143005371094,18.936622811124767,1.5646128728985786,22.21074914932251,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,63,150,10.0,199.60976886749268,19.374471263697885,1.5360757932066917,22.41091513633728,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,64,150,10.0,205.75592613220215,19.2320840153433,1.5338219478726387,23.56390881538391,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,65,150,10.0,192.3982448577881,19.40947540349214,1.5509242564439774,22.90554928779602,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,66,150,10.0,189.7183027267456,19.5658620219209,1.5356146693229675,23.833500623703003,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,67,150,10.0,197.58380317687988,19.348914385025367,1.5937729477882385,23.347342252731323,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,68,150,10.0,194.65983772277832,19.459147478940963,1.5719042718410492,23.414296865463257,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,69,150,10.0,186.7406711578369,19.639733933372007,1.527706302702427,23.59968066215515,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,70,150,10.0,192.34382820129395,19.520486524452902,1.5252604112029076,23.50745177268982,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,71,150,10.0,212.97843265533447,19.13432602696282,1.6043367087841034,23.844562768936157,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,72,150,10.0,177.23276281356812,19.87057851026124,1.5766901224851608,23.023507595062256,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,73,150,10.0,172.67132377624512,19.959645130531122,1.5944040641188622,23.281916618347168,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,74,150,10.0,210.10797691345215,19.101827746820437,1.7186110466718674,22.717442512512207,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,75,150,10.0,171.44055461883545,20.064398955729587,1.5847230553627014,22.278446674346924,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,76,150,10.0,194.18039989471436,19.477003945139852,1.5886326357722282,22.653695821762085,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,77,150,10.0,184.8071746826172,19.693833705730906,1.610305480659008,22.199661254882812,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,78,150,10.0,176.2814598083496,19.918823940928814,1.6177062913775444,22.87313485145569,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,79,150,10.0,186.19756317138672,19.644377106933522,1.6773895025253296,22.231512308120728,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,80,150,10.0,173.04176568984985,19.989311889207496,1.6447774842381477,22.793469429016113,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,81,150,10.0,166.77951097488403,20.12078981979926,1.5978920385241508,22.20487403869629,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,82,150,10.0,177.40933513641357,19.84252565770635,1.604466587305069,22.391300439834595,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,83,150,10.0,187.56576824188232,19.60411405980297,1.6413425281643867,22.688071727752686,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,84,150,10.0,164.62313508987427,20.192889218987823,1.5782289281487465,22.156715393066406,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,85,150,10.0,165.68044185638428,20.140801077688305,1.6492278352379799,22.683557987213135,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,86,150,10.0,159.7324719429016,20.3085890566727,1.6327288299798965,22.396246671676636,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,87,150,10.0,173.96030569076538,19.90263380640708,1.6502765789628029,22.918364763259888,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,88,150,10.0,151.16767168045044,20.54210232481134,1.588710866868496,22.841444730758667,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,89,150,10.0,151.34615516662598,20.521098619858083,1.5920659825205803,22.85757613182068,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,90,150,10.0,170.10539770126343,20.035856277458805,1.631488412618637,23.418977975845337,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,91,150,10.0,154.02990531921387,20.497802043107406,1.6284012347459793,22.720906734466553,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,92,150,10.0,162.1735053062439,20.24751403929048,1.6760298311710358,23.024668216705322,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,93,150,10.0,162.2310094833374,20.23353003581425,1.6726972609758377,22.581536769866943,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,94,150,10.0,176.73078203201294,19.877405983153157,1.7357294633984566,23.040013790130615,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,95,150,10.0,169.6178684234619,20.025430153572383,1.6641708463430405,22.554034948349,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,96,150,10.0,171.83164882659912,19.993847590683124,1.691560760140419,22.65778875350952,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,97,150,10.0,163.40908908843994,20.248614745781467,1.6463627070188522,23.06367564201355,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,98,150,10.0,141.36825323104858,20.82128757380783,1.6032864153385162,23.02087426185608,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,99,150,10.0,143.33438968658447,20.710414545789725,1.6520850211381912,23.720595121383667,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,100,150,10.0,140.7463879585266,20.82048858479495,1.6381006017327309,22.909340858459473,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,101,150,10.0,143.03108739852905,20.714814955791834,1.7041992172598839,23.679501056671143,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,102,150,10.0,141.67910480499268,20.75521578983202,1.6399926394224167,23.619178295135498,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,103,150,10.0,153.16044092178345,20.496929166167078,1.7224777415394783,23.475963592529297,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,104,150,10.0,139.8832974433899,20.847467490136253,1.698317289352417,23.73047399520874,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,105,150,10.0,128.3905735015869,21.229516016233173,1.6549276486039162,23.210306644439697,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,106,150,10.0,141.55389213562012,20.82284181361682,1.7054586932063103,23.738853216171265,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,107,150,10.0,151.96441221237183,20.383898278920455,1.742636226117611,22.91590404510498,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,108,150,10.0,129.4276008605957,21.25303876757065,1.654836967587471,23.591758489608765,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,109,150,10.0,153.3355450630188,20.44131366290074,1.7608824670314789,22.843791246414185,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,110,150,10.0,140.4679570198059,20.874104472960397,1.7603022307157516,22.929174423217773,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,111,150,10.0,135.67965936660767,20.97522787837366,1.738464318215847,23.576690435409546,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,112,150,10.0,144.5735263824463,20.70773414950731,1.7427054345607758,23.03432822227478,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,113,150,10.0,161.52190256118774,20.22166202396056,1.7639587074518204,23.49159336090088,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,114,150,10.0,142.37137508392334,20.76000880935886,1.775726079940796,22.95783257484436,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,115,150,10.0,128.16529870033264,21.27730841739082,1.6816172674298286,23.676251649856567,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,116,150,10.0,139.18946361541748,20.892097095879993,1.7889949977397919,22.941893100738525,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,117,150,10.0,124.44748735427856,21.34120885871544,1.7267489358782768,22.744948625564575,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,118,150,10.0,137.52435159683228,20.922531584418,1.7966705709695816,23.991820096969604,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,119,150,10.0,132.45342350006104,21.149878377122047,1.773485116660595,23.50482726097107,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,120,150,10.0,128.15565872192383,21.24763547468673,1.7483628541231155,23.660330295562744,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,121,150,10.0,139.611478805542,20.886214472872577,1.7832966819405556,23.30727458000183,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,122,150,10.0,141.2697639465332,20.80715300991916,1.7991910949349403,23.76007056236267,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,123,150,10.0,121.81045007705688,21.445807619849905,1.699307695031166,23.155982494354248,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,124,150,10.0,130.6044373512268,21.13791045250395,1.75665844976902,23.1305673122406,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,125,150,10.0,132.38040161132812,21.07369943652405,1.8003015965223312,23.510467767715454,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,126,150,10.0,135.43379402160645,21.0099706014951,1.8504435196518898,23.090473890304565,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,127,150,10.0,130.64013385772705,21.130166044810426,1.8105339854955673,23.60650634765625,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,128,150,10.0,122.01965856552124,21.424003755584643,1.7544753327965736,23.309734106063843,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,129,150,10.0,129.7198781967163,21.11750010423787,1.8228606283664703,23.44368577003479,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,130,150,10.0,134.04688262939453,21.03523985189721,1.8136849701404572,23.925583839416504,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,131,150,10.0,131.8356580734253,21.1294889386771,1.7832308188080788,23.29207420349121,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,132,150,10.0,123.72010326385498,21.41676359716802,1.7690544798970222,23.758386611938477,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,133,150,10.0,138.16042852401733,20.85922183787445,1.8665164709091187,23.0782790184021,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,134,150,10.0,130.10167837142944,21.167645666223812,1.8071175813674927,23.242480516433716,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,135,150,10.0,118.46369457244873,21.502351061408355,1.8501371815800667,22.617563724517822,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,136,150,10.0,127.02242517471313,21.238504684278368,1.8625208884477615,22.898085832595825,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,137,150,10.0,122.52700328826904,21.426381915806477,1.8137543946504593,22.369116067886353,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,138,150,10.0,133.32515954971313,21.03742954294472,1.8694257885217667,22.555596113204956,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,139,150,10.0,121.08731985092163,21.459248251382714,1.7946772053837776,22.514005422592163,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,140,150,10.0,129.2134804725647,21.24565740753367,1.9211508110165596,22.503159761428833,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,141,150,10.0,124.45774602890015,21.364747124128712,1.9548572078347206,22.938204765319824,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,142,150,10.0,122.6030535697937,21.429967929423302,1.8801392167806625,22.600131511688232,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,143,150,10.0,115.49774169921875,21.670679975345024,1.8105816021561623,22.839780807495117,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,144,150,10.0,117.85818481445312,21.585896229901714,1.8492604941129684,22.47183847427368,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,145,150,10.0,117.07393169403076,21.670140450671113,1.8435835167765617,22.528208255767822,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,146,150,10.0,116.60646772384644,21.628910944165945,1.8507669121026993,22.79869771003723,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,147,150,10.0,110.75983238220215,21.854857611144553,1.87028107047081,22.45931315422058,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,148,150,10.0,106.64383220672607,22.038316193364224,1.8445267751812935,22.939822912216187,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,149,150,10.0,115.91200542449951,21.63766895877977,1.8508959636092186,22.455626487731934,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,150,150,10.0,122.0655198097229,21.43691310549739,1.912136696279049,22.936331033706665,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\n', 'NTSCC_plus_Rayleigh': 'model,channel,epoch,epochs_target,snr_db,train_loss,train_psnr_db,estimated_bpp,epoch_seconds,train_flops_per_step,inference_flops_per_image,batch_size,crop,images_in_epoch,amp_fp16,source_commit\nNTSCC_plus,Rayleigh,1,150,10.0,41349.0400390625,-2.811202739430229,13.7027148604393,22.520040035247803,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,2,150,10.0,7017.218643188477,5.06774151405387,4.9202215522527695,22.504377365112305,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,3,150,10.0,2898.397003173828,9.29714749565262,2.0190191119909286,22.898847103118896,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,4,150,10.0,2300.8088302612305,9.656817634260607,1.536063775420189,22.53314995765686,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,5,150,10.0,1812.99951171875,10.4365035897866,1.4632453247904778,23.005796194076538,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,6,150,10.0,1475.7347564697266,11.42504783771405,1.4732406809926033,22.59389042854309,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,7,150,10.0,1145.803970336914,12.360994649190873,1.4242084845900536,22.448197603225708,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,8,150,10.0,913.327766418457,13.500498150717055,1.4107868000864983,23.122345209121704,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,9,150,10.0,834.4281883239746,13.81111831148209,1.4156472608447075,22.563008308410645,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,10,150,10.0,754.2590789794922,14.216501025711354,1.40069068223238,22.98180389404297,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,11,150,10.0,669.3445224761963,14.835019753780092,1.390677236020565,22.575226545333862,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,12,150,10.0,612.1174240112305,15.17056677156123,1.293271392583847,22.992476224899292,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,13,150,10.0,673.7242393493652,14.409002396904672,1.3593964949250221,22.571376085281372,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,14,150,10.0,568.3528594970703,15.282143925593498,1.322373628616333,22.652894496917725,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,15,150,10.0,519.9338150024414,15.680981864190137,1.3550813794136047,22.98488140106201,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,16,150,10.0,488.6125316619873,15.927141627831068,1.2736799791455269,22.435651063919067,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,17,150,10.0,493.64996337890625,15.925400125935429,1.257508173584938,23.132046699523926,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,18,150,10.0,463.74486541748047,16.184196032795903,1.2852079197764397,22.70777463912964,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,19,150,10.0,436.9586353302002,16.32518959194128,1.2925826087594032,23.13384985923767,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,20,150,10.0,444.42956161499023,16.13764852189704,1.3508744165301323,22.617018938064575,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,21,150,10.0,415.8554267883301,16.388085223557514,1.3349001854658127,22.73817729949951,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,22,150,10.0,440.98152923583984,15.982039661118373,1.3497949689626694,23.092979431152344,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,23,150,10.0,377.6946964263916,16.820958819218216,1.3299430385231972,22.490659713745117,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,24,150,10.0,377.28101539611816,16.762104942288325,1.2856604009866714,23.15365719795227,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,25,150,10.0,389.902379989624,16.555149384061647,1.3185513317584991,22.51950979232788,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,26,150,10.0,382.66830253601074,16.658107930963844,1.3070334568619728,23.168643712997437,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,27,150,10.0,403.7788715362549,16.40327289575579,1.3343323469161987,22.56281805038452,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,28,150,10.0,372.47296714782715,16.656875950861078,1.4156319797039032,22.581311464309692,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,29,150,10.0,346.21431255340576,17.062190897059132,1.4035204648971558,23.132726907730103,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,30,150,10.0,344.89992904663086,17.066271554828095,1.3389126881957054,22.53792691230774,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,31,150,10.0,331.60352325439453,17.208082876044223,1.36007309705019,22.978136777877808,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,32,150,10.0,314.33510398864746,17.441877758392156,1.3455999195575714,22.577739238739014,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,33,150,10.0,284.71247005462646,17.886455640455708,1.3179657831788063,23.037822246551514,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,34,150,10.0,325.5303955078125,17.16953216237571,1.4070902317762375,22.641459941864014,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,35,150,10.0,328.9906520843506,17.267825371166225,1.4022785797715187,22.610316038131714,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,36,150,10.0,346.92087173461914,16.961591821223177,1.4301762208342552,23.13834261894226,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,37,150,10.0,305.4383201599121,17.548619724995916,1.4165218994021416,22.63246440887451,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,38,150,10.0,310.14663887023926,17.429501413080956,1.454082764685154,23.09942364692688,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,39,150,10.0,284.72668075561523,17.88446096645657,1.4294048696756363,22.69042444229126,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,40,150,10.0,301.07518196105957,17.641418381909247,1.4636427611112595,24.007716417312622,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,41,150,10.0,305.6825075149536,17.465964763475704,1.5287245362997055,23.662906885147095,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,42,150,10.0,278.29289722442627,17.900253174723723,1.4857235476374626,23.97583556175232,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,43,150,10.0,274.63812255859375,17.932486286102566,1.493614099919796,24.351113080978394,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,44,150,10.0,267.4329891204834,18.05404308061473,1.4617724642157555,23.551116466522217,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,45,150,10.0,325.36741638183594,17.227904780618747,1.4806938841938972,23.90597367286682,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,46,150,10.0,272.99556827545166,17.987910275977356,1.4714614301919937,23.138901472091675,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,47,150,10.0,261.4921474456787,18.046102900437244,1.5114870965480804,22.986191987991333,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,48,150,10.0,236.07163524627686,18.568698300579804,1.4717905521392822,23.547114610671997,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,49,150,10.0,251.1828498840332,18.24987937641717,1.5031865313649178,22.703654766082764,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,50,150,10.0,249.5067834854126,18.26558696009133,1.4935062527656555,24.371196031570435,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,51,150,10.0,262.0569877624512,18.082113963900518,1.487262561917305,23.106560707092285,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,52,150,10.0,267.0299777984619,17.98417243165192,1.5362978354096413,23.71036124229431,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,53,150,10.0,254.35997772216797,18.184364911683012,1.5332511737942696,23.592999935150146,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,54,150,10.0,225.9652156829834,18.748546326644824,1.4918580874800682,24.25970983505249,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,55,150,10.0,219.85447883605957,18.883265070593048,1.4734684079885483,24.31025457382202,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,56,150,10.0,228.6451873779297,18.5803494160967,1.4965251609683037,24.172194480895996,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,57,150,10.0,249.8910894393921,18.38297959066064,1.5353806018829346,24.150861501693726,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,58,150,10.0,226.99152946472168,18.666271856772795,1.4821219742298126,24.176467895507812,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,59,150,10.0,227.79400539398193,18.55074449669711,1.5872857794165611,24.20240879058838,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,60,150,10.0,238.77710628509521,18.44924824477762,1.7685507982969284,23.737866163253784,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,61,150,10.0,245.19473934173584,18.325651807167255,1.9795779213309288,24.429202556610107,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,62,150,10.0,250.53323650360107,18.214239872065875,1.8453193008899689,23.235646724700928,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,63,150,10.0,219.3070936203003,18.892167553290676,1.7090303376317024,23.576057195663452,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,64,150,10.0,211.44362449645996,19.089178279321043,1.6466707661747932,24.090883016586304,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,65,150,10.0,206.44568920135498,19.027847223873106,1.622465193271637,23.68265652656555,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,66,150,10.0,195.32946968078613,19.38799941453905,1.606980837881565,23.845715761184692,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,67,150,10.0,207.91192531585693,19.063302022126532,1.6272514536976814,23.5789315700531,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,68,150,10.0,198.55445194244385,19.267390522692477,1.6195780709385872,23.131535053253174,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,69,150,10.0,202.18408679962158,19.24887362928481,1.6397320106625557,23.247233867645264,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,70,150,10.0,228.55742168426514,18.639176823946986,1.6214455664157867,22.91496777534485,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,71,150,10.0,202.5975513458252,19.187888974415728,1.6013498529791832,23.368141651153564,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,72,150,10.0,183.78849601745605,19.59604157783565,1.589103676378727,22.80917453765869,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,73,150,10.0,182.43411779403687,19.663446503285154,1.619361698627472,23.32826042175293,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,74,150,10.0,212.27516651153564,18.95482872624664,1.6736816689372063,22.852824926376343,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,75,150,10.0,194.80434799194336,19.354767287816426,1.6313253343105316,22.853309392929077,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,76,150,10.0,187.6400489807129,19.47294150981587,1.6123105138540268,23.47062611579895,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,77,150,10.0,197.1684808731079,19.25924460462,1.6235572770237923,22.855468034744263,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,78,150,10.0,196.21375560760498,19.294894251718258,1.625948190689087,23.60096549987793,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,79,150,10.0,192.69738483428955,19.334900762112483,1.667699284851551,22.866288661956787,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,80,150,10.0,191.09291076660156,19.36888022043973,1.672297403216362,23.381592988967896,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,81,150,10.0,196.45355224609375,19.297133166731886,1.6853880807757378,22.7796151638031,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,82,150,10.0,185.26821327209473,19.542932578510204,1.6442170590162277,23.370110750198364,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,83,150,10.0,178.98089504241943,19.689110932086344,1.6860781013965607,22.871166229248047,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,84,150,10.0,167.5806064605713,19.974423083858703,1.6480986550450325,22.865784645080566,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,85,150,10.0,164.46697282791138,20.04796682484536,1.6459848135709763,23.367358684539795,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,86,150,10.0,169.43793869018555,19.9550010184008,1.644448071718216,22.848153352737427,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,87,150,10.0,171.09604406356812,19.936118005019612,1.7138966470956802,23.431975603103638,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,88,150,10.0,150.2711844444275,20.530665976924773,1.64288280159235,22.806061506271362,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,89,150,10.0,163.21251726150513,20.126881599039574,1.684489518404007,23.59006690979004,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,90,150,10.0,180.39251708984375,19.579387611177708,1.726236216723919,23.578407526016235,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,91,150,10.0,158.18418741226196,20.221396613911192,1.7174258455634117,23.35446572303772,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,92,150,10.0,172.891685962677,19.818228001338213,1.723371036350727,24.07010269165039,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,93,150,10.0,166.05180263519287,20.01789594871554,1.732001043856144,24.01545810699463,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,94,150,10.0,174.0342583656311,19.83910702318102,1.7641351372003555,24.1419939994812,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,95,150,10.0,167.27905797958374,19.99489153402269,1.7082859948277473,23.822604179382324,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,96,150,10.0,176.7899842262268,19.702055700244014,1.7524599134922028,23.727959632873535,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,97,150,10.0,156.2618489265442,20.26346293986687,1.701586239039898,24.790911197662354,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,98,150,10.0,161.31925344467163,20.195867437851234,1.6937437281012535,23.91642999649048,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,99,150,10.0,160.12151765823364,20.17072427959927,1.717956893146038,24.512569904327393,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,100,150,10.0,156.79281520843506,20.253307262131592,1.7448887303471565,24.146832704544067,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,101,150,10.0,153.26374340057373,20.296931506998995,1.7644638940691948,24.066624641418457,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,102,150,10.0,143.90242338180542,20.612143662859445,1.6879815012216568,24.76875114440918,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,103,150,10.0,173.11160707473755,19.85380774188596,1.7996005341410637,23.172399044036865,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,104,150,10.0,157.00701904296875,20.21528786252141,1.742096669971943,23.64422082901001,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,105,150,10.0,161.85227823257446,20.097117431876725,1.775273971259594,23.036388158798218,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,106,150,10.0,155.01375007629395,20.321662252170224,1.7341657429933548,23.619014501571655,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,107,150,10.0,155.92805528640747,20.235085086546324,1.7419947013258934,22.92130208015442,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,108,150,10.0,143.45263528823853,20.59955849898559,1.7072131261229515,23.950132369995117,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,109,150,10.0,162.60852813720703,20.065066163614794,1.8151395991444588,22.99645161628723,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,110,150,10.0,143.83264636993408,20.585233918708514,1.7419798970222473,22.88100004196167,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,111,150,10.0,163.5464949607849,20.027948546052706,1.7913448363542557,23.47038722038269,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,112,150,10.0,155.60212230682373,20.227710063101224,1.7372583746910095,23.075082302093506,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,113,150,10.0,154.15455198287964,20.286948219900097,1.7593494430184364,23.41478419303894,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,114,150,10.0,146.2304925918579,20.535696592598267,1.8013902604579926,22.942867755889893,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,115,150,10.0,139.64932918548584,20.779812072028818,1.8003129735589027,22.959189414978027,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,116,150,10.0,154.157555103302,20.31603569275451,1.8626248091459274,23.564975261688232,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,117,150,10.0,133.95377349853516,20.911204898028192,1.7704730033874512,22.919907808303833,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,118,150,10.0,150.5819444656372,20.333507702675742,1.8353237882256508,23.605074405670166,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,119,150,10.0,145.16802597045898,20.61039884706495,1.841575987637043,23.332337617874146,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,120,150,10.0,135.9186191558838,20.839691191059913,1.8492376655340195,23.71750521659851,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,121,150,10.0,139.28937196731567,20.65660622629622,1.7906989455223083,23.059417009353638,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,122,150,10.0,163.91792964935303,19.953561794081306,1.9349487647414207,23.220706939697266,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,123,150,10.0,138.60392332077026,20.765608658398715,1.82143634557724,23.528882265090942,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,124,150,10.0,133.88504028320312,20.867889361562707,1.839795008301735,22.943483352661133,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,125,150,10.0,138.42737340927124,20.735092604284393,1.8936958238482475,23.325660943984985,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,126,150,10.0,143.6514801979065,20.594762394671307,1.908023625612259,22.9375262260437,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,127,150,10.0,141.0035433769226,20.647970288188354,1.9156325161457062,23.49112606048584,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,128,150,10.0,143.24181747436523,20.628955583508795,1.8632602095603943,22.958664178848267,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,129,150,10.0,153.42424726486206,20.363721442840536,1.870009072124958,22.911504983901978,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,130,150,10.0,162.60843992233276,20.063234249712153,1.8844954147934914,23.420605897903442,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,131,150,10.0,146.35474586486816,20.54873132350144,1.9010154828429222,23.02019238471985,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,132,150,10.0,134.35641956329346,20.910001897516214,1.8779368624091148,23.42356848716736,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,133,150,10.0,138.3089485168457,20.771583320387368,1.8833899348974228,22.864949703216553,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,134,150,10.0,126.99617195129395,21.17065585983637,1.8714429661631584,22.94248104095459,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,135,150,10.0,120.32891654968262,21.323804717487064,1.8223903700709343,23.30249786376953,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,136,150,10.0,146.49135828018188,20.513073026513233,1.9494082033634186,23.075963735580444,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,137,150,10.0,137.79952096939087,20.816924791852884,1.8637294322252274,23.638261556625366,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,138,150,10.0,135.10473537445068,20.903240081781952,1.9143378362059593,23.570582628250122,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,139,150,10.0,136.50968742370605,20.84909200503125,1.9048939421772957,23.885743141174316,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,140,150,10.0,143.03124856948853,20.512204840308694,1.9223804846405983,23.614594221115112,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,141,150,10.0,126.23761177062988,21.164619427343553,1.9240339547395706,24.37408971786499,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,142,150,10.0,131.61556482315063,20.994203899306484,1.9329662695527077,24.008030652999878,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,143,150,10.0,128.6069564819336,21.072502662627382,1.8831132799386978,23.878161430358887,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,144,150,10.0,127.77489233016968,21.049748298532414,1.8845093473792076,23.60043454170227,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,145,150,10.0,132.05419874191284,20.974157946849317,1.9183058738708496,24.060317039489746,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,146,150,10.0,121.14570236206055,21.322975073263635,1.9214284345507622,24.627951860427856,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,147,150,10.0,127.45696353912354,21.093909485954157,1.974757820367813,23.712194442749023,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,148,150,10.0,121.36186170578003,21.27488364495281,1.909542664885521,24.261477947235107,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,149,150,10.0,127.34014701843262,21.162953981265964,1.8894806876778603,23.461796522140503,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,150,150,10.0,124.92308855056763,21.20550629251973,1.937358982861042,23.61973810195923,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\n', 'SwinJSCC_AWGN': 'model,channel,snr_db,epoch,train_loss,epoch_seconds,samples_seen,div2k_pool_images,crop_px,batch_size,rate_C,nominal_cbr_target,training_flops_per_sample,inference_flops_per_image_256,source_commit\nSwinJSCC,AWGN,10,1,5914.508173828125,31.922123193740845,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,2,1715.5396020507812,31.240333795547485,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,3,1305.0283264160157,33.470678091049194,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,4,1174.3539599609376,33.053022384643555,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,5,901.09833984375,32.050657987594604,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,6,731.4704119873047,32.640095949172974,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,7,665.7782537841797,32.28849124908447,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,8,579.8739422607422,32.08848857879639,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,9,568.1187280273438,32.75532388687134,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,10,528.0645062255859,32.61786770820618,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,11,501.0079595947266,32.27299213409424,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,12,493.7427819824219,32.205572843551636,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,13,518.431362915039,32.10867881774902,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,14,468.02425170898437,32.07184553146362,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,15,516.7819219970703,32.03237986564636,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,16,472.7680010986328,32.40301823616028,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,17,431.7528326416016,32.18567419052124,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,18,417.24348815917966,32.0335533618927,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,19,392.5103680419922,32.0800347328186,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,20,369.5139398193359,32.36993956565857,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,21,320.59539672851565,32.11539363861084,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,22,337.72553283691406,32.28520321846008,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,23,318.04296112060547,32.562000036239624,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,24,303.47480346679686,32.82436275482178,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,25,300.6097479248047,32.72264051437378,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,26,306.6395962524414,32.68882346153259,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,27,292.9251638793945,32.98806142807007,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,28,284.2174450683594,32.790262937545776,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,29,293.18963317871095,32.47905445098877,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,30,286.1163150024414,32.563809871673584,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,31,267.0064300537109,32.552616119384766,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,32,262.6916387939453,32.64459538459778,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,33,269.3181021118164,32.437723875045776,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,34,272.36627380371095,32.62664556503296,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,35,272.5655953979492,32.44706082344055,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,36,252.2055465698242,31.92648458480835,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,37,247.3036001586914,31.917221546173096,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,38,239.01985229492186,32.22249913215637,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,39,244.091962890625,31.95232892036438,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,40,236.50627990722657,31.888908624649048,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,41,241.35210418701172,31.90527057647705,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,42,269.06991912841795,32.3228600025177,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,43,247.99042053222655,31.924785614013672,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,44,235.08724609375,31.99418067932129,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,45,227.47603958129883,32.787209033966064,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,46,236.8067413330078,32.71818780899048,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,47,242.85546478271485,32.56499528884888,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,48,248.93726654052733,32.762781858444214,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,49,260.01149597167966,32.66610312461853,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,50,231.67631072998046,33.13443207740784,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,51,224.1628384399414,32.7195041179657,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,52,227.80715148925782,32.73602032661438,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,53,238.06722900390625,33.014870166778564,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,54,223.0051412963867,32.68930220603943,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,55,215.75476287841798,32.87925624847412,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,56,212.47706268310546,32.92984366416931,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,57,217.26709564208986,32.966163635253906,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,58,214.2413134765625,32.96325874328613,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,59,215.05708892822267,33.02809238433838,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,60,209.82993072509765,32.59765076637268,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,61,206.6953219604492,33.14355254173279,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,62,209.33927001953126,32.900224447250366,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,63,223.6236087036133,32.55103063583374,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,64,221.26095947265625,32.993154525756836,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,65,205.5710415649414,32.87488579750061,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,66,202.49397369384766,32.743446826934814,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,67,203.08588623046876,33.02867293357849,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,68,419.871806640625,33.129454374313354,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,69,262.36897247314454,32.74918055534363,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,70,211.94059967041017,33.06981420516968,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,71,202.36114349365235,32.89937090873718,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,72,195.75191940307616,33.23308801651001,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,73,192.05144775390625,32.789172649383545,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,74,195.32590209960938,33.038044452667236,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,75,190.4113639831543,32.664226770401,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,76,194.27890991210938,33.032925844192505,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,77,191.59714294433593,32.87955832481384,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,78,185.71644821166993,32.08358073234558,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,79,185.6295309448242,32.536057472229004,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,80,184.0677325439453,31.944904327392578,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,81,184.50315826416016,32.17629623413086,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,82,181.78978881835937,32.11675024032593,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,83,184.4925924682617,32.30310297012329,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,84,183.4310452270508,32.13975501060486,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,85,180.67284423828124,32.287065267562866,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,86,188.2186880493164,32.03623867034912,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,87,200.1097721862793,32.77889609336853,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,88,194.6270571899414,32.87257218360901,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,89,176.86801300048828,32.923259019851685,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,90,172.09141326904296,32.92894983291626,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,91,178.12488647460938,32.887765645980835,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,92,182.28135314941406,32.99528956413269,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,93,182.9743588256836,32.96932053565979,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,94,187.1357551574707,33.040308713912964,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,95,197.64331237792967,32.91528367996216,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,96,196.58222885131835,32.75092816352844,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,97,173.84477340698243,32.86054086685181,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,98,172.37880874633788,33.09218740463257,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,99,172.38588592529297,32.7857460975647,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,100,174.42447784423828,33.06733584403992,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,101,167.76729705810547,33.01959204673767,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,102,167.84351776123046,33.13789176940918,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,103,169.66317901611328,33.09324359893799,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,104,171.79619140625,32.76684355735779,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,105,168.11501419067383,33.23146176338196,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,106,169.11772506713868,32.81329870223999,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,107,164.25330261230468,32.94566011428833,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,108,168.20700805664063,32.91733407974243,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,109,164.0717385864258,32.93465709686279,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,110,166.7588789367676,33.10228371620178,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,111,168.65107467651367,33.008015632629395,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,112,173.46021423339843,32.92093086242676,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,113,180.90802215576173,33.12701344490051,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,114,197.9430368041992,32.92010021209717,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,115,172.69794494628906,32.735689878463745,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,116,164.52147216796874,33.07982802391052,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,117,159.73559829711914,32.817352056503296,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,118,157.98093338012694,32.82981991767883,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,119,160.09652084350586,32.873761892318726,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,120,160.3867349243164,33.02677035331726,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,121,163.80256195068358,32.060847759246826,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,122,160.93255905151366,31.93729591369629,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,123,166.7895556640625,32.04640865325928,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,124,173.98361892700194,32.19482064247131,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,125,169.9283497619629,32.041792154312134,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,126,158.33031143188475,32.117156982421875,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,127,157.48911193847655,32.082250118255615,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,128,165.78709716796874,32.217265367507935,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,129,177.6291046142578,32.1487078666687,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,130,158.6059539794922,32.804683685302734,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,131,184.5863380432129,33.00754189491272,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,132,157.6952732849121,32.957663774490356,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,133,152.41377731323243,32.842952728271484,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,134,159.08254913330077,33.34526968002319,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,135,148.39089935302735,33.08551836013794,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,136,149.65462173461913,33.51032471656799,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,137,150.4513624572754,33.13310647010803,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,138,148.28859878540038,32.919225454330444,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,139,147.90749938964845,33.3720281124115,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,140,152.2922932434082,33.20817255973816,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,141,148.43096771240235,33.33791375160217,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,142,150.69519729614257,33.791733741760254,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,143,148.9765074157715,33.27668213844299,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,144,150.7433654785156,33.62900400161743,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,145,143.98954162597656,33.15772271156311,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,146,143.2365983581543,33.91983723640442,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,147,149.39092803955077,33.57349896430969,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,148,180.17995239257812,33.211806297302246,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,149,181.3255601501465,33.48083686828613,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,AWGN,10,150,148.25152603149414,33.526106119155884,800,800,256,16,96,0.0625,206543376388,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\n', 'SwinJSCC_Rayleigh': 'model,channel,snr_db,epoch,train_loss,epoch_seconds,samples_seen,div2k_pool_images,crop_px,batch_size,rate_C,nominal_cbr_target,training_flops_per_sample,inference_flops_per_image_256,source_commit\nSwinJSCC,Rayleigh,10,1,6117.497758789063,33.00460147857666,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,2,2940.45404296875,32.666401386260986,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,3,1450.6644970703126,32.3305025100708,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,4,1107.0851391601564,32.81292104721069,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,5,845.4801391601562,32.4564528465271,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,6,785.4969616699219,32.744664430618286,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,7,712.0084826660157,32.75075364112854,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,8,661.4839770507813,32.70741391181946,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,9,604.6363739013672,32.48121523857117,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,10,579.2678112792969,32.853952169418335,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,11,568.6890606689453,32.57845211029053,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,12,542.3505987548829,32.53321123123169,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,13,539.2571099853516,32.89191293716431,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,14,536.2192517089844,32.542887687683105,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,15,526.1584344482421,32.79696345329285,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,16,515.7994219970703,32.577264070510864,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,17,512.141806640625,33.12564754486084,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,18,455.4456500244141,32.704946756362915,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,19,427.0693115234375,32.52392840385437,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,20,418.0719360351562,32.87265491485596,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,21,417.7926934814453,32.908135175704956,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,22,407.22285888671877,32.68833613395691,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,23,413.3738098144531,32.51145911216736,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,24,399.9777258300781,32.516350507736206,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,25,379.6293817138672,32.602060317993164,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,26,378.7474295043945,32.70930314064026,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,27,378.81779907226564,32.745752811431885,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,28,362.34194213867187,32.455451011657715,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,29,366.35482421875,33.09481644630432,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,30,370.03554870605467,32.62946701049805,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,31,368.64172119140625,32.77213215827942,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,32,348.4305786132812,32.86175203323364,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,33,347.1874917602539,32.736188650131226,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,34,351.9883139038086,32.5357449054718,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,35,361.3121127319336,32.83746957778931,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,36,362.3460333251953,32.904685497283936,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,37,376.5456884765625,32.41465425491333,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,38,348.6587615966797,32.87017774581909,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,39,336.12737213134767,32.39112997055054,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,40,330.2264782714844,33.03760886192322,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,41,328.00126403808594,32.580132246017456,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,42,329.7328793334961,32.50690579414368,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,43,379.9675439453125,32.922905921936035,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,44,351.5870361328125,32.715606689453125,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,45,323.27275787353517,32.63901662826538,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,46,314.79784210205077,32.82997679710388,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,47,312.60571716308596,32.931727170944214,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,48,312.986513671875,32.64475131034851,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,49,316.8612469482422,32.74171996116638,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,50,309.8586178588867,32.669267654418945,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,51,303.9004284667969,32.843982219696045,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,52,304.13493408203124,32.81002163887024,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,53,304.23478057861325,32.668766260147095,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,54,329.0342984008789,32.82355260848999,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,55,315.2626922607422,32.898592710494995,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,56,302.8127032470703,32.520671129226685,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,57,294.6569583129883,32.704370737075806,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,58,314.63295684814454,32.69094896316528,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,59,328.43602447509767,32.54195976257324,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,60,305.1177920532227,32.89021873474121,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,61,294.4235482788086,32.463502168655396,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,62,288.6850250244141,32.94533371925354,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,63,303.76115844726564,32.64926314353943,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,64,309.2133483886719,32.37804079055786,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,65,289.86264770507813,32.84104299545288,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,66,291.82790802001955,32.92083430290222,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,67,311.86508880615236,32.618892669677734,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,68,322.9373776245117,32.639055013656616,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,69,292.8161978149414,32.88741993904114,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,70,288.15659606933593,32.65854024887085,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,71,281.8223745727539,32.794198513031006,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,72,288.00409332275393,32.920302391052246,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,73,281.4549383544922,32.7976930141449,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,74,289.71152557373046,32.552454471588135,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,75,283.3974203491211,32.54139566421509,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,76,274.53971771240236,32.818963050842285,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,77,293.04407684326173,32.8353853225708,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,78,288.54009216308594,32.71301817893982,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,79,287.48545623779296,32.72706460952759,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,80,279.52194610595706,32.77929663658142,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,81,291.0263751220703,33.00232982635498,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,82,286.95116821289065,32.84346842765808,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,83,275.72733337402343,32.60620307922363,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,84,280.4857723999023,32.74976563453674,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,85,276.4055819702148,32.69918084144592,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,86,288.09898376464844,32.705326795578,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,87,292.99851806640623,32.59932088851929,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,88,274.1498422241211,32.984341621398926,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,89,275.3171142578125,32.57872009277344,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,90,281.98453125,32.71767330169678,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,91,297.3554977416992,32.50469255447388,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,92,284.3489712524414,32.896862745285034,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,93,267.9740493774414,32.52770781517029,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,94,266.88426513671874,32.65934467315674,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,95,269.6381918334961,32.64648151397705,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,96,269.93113037109373,32.74088931083679,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,97,272.77084564208985,32.7926299571991,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,98,276.00264251708984,32.781107902526855,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,99,265.2820104980469,32.97398900985718,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,100,266.43761840820315,32.589000940322876,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,101,263.03787689208986,32.80697059631348,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,102,267.7657275390625,32.6957585811615,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,103,263.70043701171875,32.98340940475464,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,104,263.913229675293,32.75555682182312,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,105,259.6871469116211,32.64498734474182,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,106,269.65345275878906,32.60702085494995,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,107,288.0998794555664,33.10315656661987,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,108,268.5603060913086,32.60263800621033,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,109,264.64624786376953,32.811028242111206,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,110,260.60961853027345,33.0436315536499,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,111,260.07462432861325,32.74812650680542,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,112,257.4843899536133,32.65305805206299,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,113,258.6748583984375,32.73365378379822,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,114,256.0460314941406,32.82154083251953,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,115,261.8252651977539,32.84092378616333,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,116,294.83243957519534,32.73822641372681,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,117,261.84583587646483,32.6727032661438,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,118,256.06024871826173,33.023761510849,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,119,258.4250369262695,32.65232610702515,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,120,260.6511447143555,32.728118896484375,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,121,247.95060485839844,32.97149395942688,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,122,248.1572036743164,32.85073637962341,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,123,262.3545819091797,32.59599423408508,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,124,280.95799713134767,32.66242957115173,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,125,312.4850469970703,32.78427028656006,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,126,257.43538970947264,32.52226996421814,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,127,244.97424438476563,32.48807144165039,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,128,249.15579559326173,32.735620737075806,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,129,258.7464797973633,32.80459237098694,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,130,258.1644256591797,32.76947259902954,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,131,244.38081604003906,32.73301577568054,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,132,240.97129669189454,32.5749306678772,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,133,241.17951599121093,32.92505598068237,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,134,243.9506463623047,32.47197484970093,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,135,245.8376892089844,32.58714699745178,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,136,239.41829376220704,32.82696795463562,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,137,245.02794647216797,32.79572033882141,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,138,259.1303094482422,32.53009867668152,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,139,271.88142639160156,32.630277156829834,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,140,237.2550555419922,32.77623391151428,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,141,237.5701754760742,32.587005853652954,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,142,241.39352142333985,32.61740446090698,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,143,237.9871060180664,32.62487077713013,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,144,250.36004455566408,32.96952795982361,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,145,243.54837463378905,32.57476830482483,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,146,240.74395294189452,32.63235306739807,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,147,248.57347259521484,32.516371726989746,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,148,249.94560272216796,32.84753727912903,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,149,233.4350442504883,32.568480253219604,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\nSwinJSCC,Rayleigh,10,150,239.47413024902343,32.713998317718506,800,800,256,16,96,0.0625,206543413252,68925788161,a6d0e6da53548976acbe9317839a077ef31f190f\n', 'DeepJSCC_Q_AWGN': 'model,channel,snr_db,epoch,train_mse,epoch_seconds,samples_seen,div2k_pool_images,crop_px,batch_size,encoder_decoder_source_commit,complex_channel_uses_per_pixel_rate,constellation_size,training_flops_per_sample,inference_flops_per_image_128\r\nDeepJSCC-Q (paper-based),AWGN,10.0,1,0.0945489609241485,2.3548903465271,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,2,0.0483537125587463,1.1941559314727783,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,3,0.03460806556046,1.1882688999176023,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,4,0.0301053823530673,1.267920732498169,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,5,0.0275925862044096,1.1734542846679688,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,6,0.0257469484210014,1.205577373504639,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,7,0.0247499033808708,1.1700491905212402,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,8,0.0235024623572826,1.1681644916534424,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,9,0.0212335641682148,1.177741527557373,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,10,0.0203045272454619,1.190654993057251,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,11,0.0181330560520291,1.1921250820159912,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,12,0.0171803518384695,1.2116825580596924,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,13,0.016158338189125,1.1637365818023682,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,14,0.0150549080967903,1.167385816574097,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,15,0.0148752744495868,1.172964334487915,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,16,0.014264973513782,1.1853759288787842,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,17,0.0135902126878499,1.2100121974945068,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,18,0.0135630049183964,1.1728460788726809,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,19,0.0135153041034936,1.165677785873413,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,20,0.0127252108231186,1.1983435153961182,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,21,0.0126683831959962,1.2439024448394775,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,22,0.0124561695381999,1.1612722873687744,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,23,0.0121577448770403,1.1728458404541016,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,24,0.0118387313559651,1.1759915351867676,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,25,0.0116713551431894,1.173635959625244,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,26,0.0116915054991841,1.1735587120056152,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,27,0.0115945532172918,1.1473288536071775,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,28,0.0112721099331974,1.1825032234191897,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,29,0.0111443894356489,1.2279019355773926,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,30,0.0112089437246322,1.1664183139801023,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,31,0.0111604043096303,1.165130853652954,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,32,0.0108110455796122,1.177161455154419,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,33,0.0111283216252923,1.157717227935791,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,34,0.010699095763266,1.1748614311218262,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,35,0.0100342205166816,1.176086664199829,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,36,0.0102678679674863,1.1603190898895264,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,37,0.0107071175798773,1.207927703857422,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,38,0.0101368808187544,1.1943387985229492,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,39,0.0101012555137276,1.1777586936950684,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,40,0.0101544739492237,1.1741690635681152,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,41,0.0097038545832037,1.167776584625244,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,42,0.0099863182939589,1.1592671871185305,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,43,0.0099544354341924,1.173551321029663,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,44,0.0100491126999259,1.1926140785217283,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,45,0.0094402954913675,1.1535086631774902,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,46,0.0096906349062919,1.2114522457122805,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,47,0.0094904478639364,1.1705727577209473,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,48,0.0094748592376708,1.1772265434265137,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,49,0.009555081911385,1.1854290962219238,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,50,0.0092837292142212,1.170155048370361,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,51,0.0093904864415526,1.1718976497650146,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,52,0.0091707900166511,1.170356273651123,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,53,0.0091910100728273,1.195260524749756,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,54,0.0091785761341452,1.2118897438049316,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,55,0.0092401767149567,1.1817295551300049,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,56,0.008912780676037,1.1785831451416016,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,57,0.0090893474593758,1.1763694286346436,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,58,0.0091103510931134,1.1707556247711182,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,59,0.0089059867151081,1.1792640686035156,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,60,0.0089342444948852,1.1835880279541016,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,61,0.0089074667915701,1.181077003479004,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,62,0.0087606184743344,1.1522457599639893,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,63,0.0086119010858237,1.1845288276672363,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,64,0.0085183637402951,1.1657037734985352,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,65,0.0083611854352056,1.187464475631714,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,66,0.0083299803175032,1.1889235973358154,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,67,0.0082088368572294,1.176793336868286,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,68,0.0084713934734463,1.175894021987915,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,69,0.0085587044991552,1.190126895904541,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,70,0.0082769168540835,1.1784570217132568,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,71,0.0082224120758473,1.2374050617218018,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,72,0.0082562523148953,1.1744632720947266,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,73,0.0083555749058723,1.179513692855835,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,74,0.0082541667483747,1.1923556327819824,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,75,0.0084209889359772,1.1598787307739258,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,76,0.0083611300960183,1.1808185577392578,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,77,0.0083676541037857,1.189856767654419,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,78,0.0082846472039818,1.1801769733428955,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,79,0.0081698591634631,1.2278330326080322,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,80,0.0082873390056192,1.1655104160308838,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,81,0.0081713656708598,1.1573536396026611,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,82,0.0080343118309974,1.152953863143921,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,83,0.0081857145950198,1.1752924919128418,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,84,0.007880520503968,1.1766691207885742,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,85,0.0078222645819187,1.151933670043945,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,86,0.0079186275601387,1.1848242282867432,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,87,0.008003021813929,1.1687052249908447,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,88,0.0080730932764709,1.2232701778411863,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,89,0.0076545989699661,1.1613333225250244,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,90,0.0079655142687261,1.1816635131835938,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,91,0.0076951887272298,1.1902179718017578,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,92,0.0076349351182579,1.1637234687805176,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,93,0.0078377331234514,1.2058372497558594,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,94,0.007562307678163,1.1918513774871826,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,95,0.0075895594991743,1.177964210510254,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,96,0.0074595113284885,1.2038123607635498,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,97,0.0074726503901183,1.1696031093597412,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,98,0.0073461151495575,1.174595832824707,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,99,0.0074568420834839,1.154270887374878,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,100,0.0070386039838194,1.1717472076416016,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,101,0.0073096751980483,1.1778712272644043,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,102,0.007380922101438,1.1788890361785889,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,103,0.0071595295332372,1.1620073318481443,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,104,0.0068986983038485,1.171379327774048,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,105,0.0073166277445852,1.1459903717041016,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,106,0.0069997107237577,1.162975549697876,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,107,0.007032283693552,1.164934158325195,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,108,0.0070743139646947,1.177534103393555,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,109,0.0072403037548065,1.157847881317139,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,110,0.0069017297774553,1.1822376251220703,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,111,0.006910025216639,1.174558401107788,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,112,0.0067075589485466,1.1598126888275146,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,113,0.0067250325717031,1.2017822265625,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,114,0.0066744121536612,1.1654701232910156,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,115,0.0067351737245917,1.1665711402893066,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,116,0.0067183930799365,1.162792444229126,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,117,0.0067408776655793,1.168388843536377,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,118,0.0067482607252895,1.1702313423156738,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,119,0.0064450632221996,1.1828477382659912,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,120,0.0067110979557037,1.1621975898742676,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,121,0.0064787874743342,1.2395596504211426,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,122,0.0063387360051274,1.168717622756958,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,123,0.0065582640469074,1.2046396732330322,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,124,0.0066587164811789,1.184309482574463,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,125,0.0064393299445509,1.165266752243042,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,126,0.0063750675506889,1.1662085056304932,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,127,0.006396318282932,1.204158067703247,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,128,0.0064254466630518,1.186065912246704,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,129,0.0063955527544021,1.1776206493377686,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,130,0.0065847012773156,1.184218168258667,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,131,0.0064557681046426,1.1389048099517822,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,132,0.0064574759826064,1.161302089691162,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,133,0.0063432992249727,1.159961462020874,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,134,0.0062863172963261,1.153904914855957,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,135,0.006412803325802,1.1562395095825195,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,136,0.0065800989419221,1.1584415435791016,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,137,0.0067523053660988,1.149644136428833,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,138,0.0064930722862482,1.1844611167907717,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,139,0.0064140028506517,1.1491727828979492,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,140,0.0060540720820426,1.1588354110717771,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,141,0.0065500956401228,1.1802663803100586,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,142,0.0060990187712013,1.1273157596588137,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,143,0.0060526865534484,1.1575078964233398,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,144,0.0061629670113325,1.125842809677124,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,145,0.006132460925728,1.135373830795288,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,146,0.0063900927267968,1.1166183948516846,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,147,0.0060239511914551,1.1941368579864502,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,148,0.0060263780690729,1.1600356101989746,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,149,0.0060230988822877,1.1399829387664795,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\nDeepJSCC-Q (paper-based),AWGN,10.0,150,0.006289195138961,1.1511707305908203,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168034406,167878658\r\n', 'DeepJSCC_Q_Rayleigh': 'model,channel,snr_db,epoch,train_mse,epoch_seconds,samples_seen,div2k_pool_images,crop_px,batch_size,encoder_decoder_source_commit,complex_channel_uses_per_pixel_rate,constellation_size,training_flops_per_sample,inference_flops_per_image_128\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,1,0.0965042477846145,1.2149174213409424,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,2,0.0479013583064079,1.2105798721313477,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,3,0.0342023898661136,1.2141098976135254,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,4,0.0298537141084671,1.2055201530456543,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,5,0.0279372901469469,1.2190613746643066,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,6,0.0266744013875722,1.2221837043762207,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,7,0.0259154673665761,1.1998717784881592,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,8,0.0245663902163505,1.213010311126709,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,9,0.0234175749868154,1.2086257934570312,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,10,0.0221945948898792,1.2092490196228027,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,11,0.0196351426839828,1.2361435890197754,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,12,0.018604578115046,1.235872745513916,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,13,0.0179463067650795,1.2112789154052734,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,14,0.0163199960812926,1.226779222488403,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,15,0.0158381794393062,1.2015986442565918,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,16,0.0150116916000843,1.2026114463806152,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,17,0.015128620006144,1.19808030128479,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,18,0.0149923112615942,1.2249808311462402,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,19,0.0146766732633113,1.274282693862915,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,20,0.014310656003654,1.1855242252349854,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,21,0.013722020201385,1.2024216651916504,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,22,0.0133473275601863,1.19575834274292,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,23,0.0131739649921655,1.238964557647705,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,24,0.0131335811689496,1.2062506675720217,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,25,0.01386892657727,1.209670066833496,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,26,0.012699350900948,1.1976053714752195,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,27,0.0124631018191576,1.2505691051483154,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,28,0.0126579153910279,1.1906113624572754,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,29,0.0123959788307547,1.202868938446045,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,30,0.012703414671123,1.2056357860565186,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,31,0.0118126131594181,1.2061309814453125,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,32,0.0124459052085876,1.1872479915618896,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,33,0.0121904730796813,1.2069284915924072,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,34,0.0122679741680622,1.2064166069030762,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,35,0.011765486523509,1.2302839756011963,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,36,0.0114662681519985,1.190854787826538,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,37,0.0112111511453986,1.2088096141815186,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,38,0.0111393445730209,1.212766408920288,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,39,0.0112310319766402,1.2006094455718994,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,40,0.0112560972571372,1.1917800903320312,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,41,0.0112363113835453,1.1874158382415771,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,42,0.0111522372812032,1.1905136108398438,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,43,0.0109755032137036,1.2283542156219482,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,44,0.0107946536317467,1.196237087249756,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,45,0.0111752483621239,1.2188997268676758,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,46,0.0108955758810043,1.2132952213287354,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,47,0.0109979228302836,1.178325891494751,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,48,0.0109952720627188,1.197126388549805,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,49,0.0105166415125131,1.2400779724121094,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,50,0.0105701799690723,1.224665880203247,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,51,0.0108159065619111,1.2237355709075928,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,52,0.0102876744046807,1.2222635746002195,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,53,0.0104475062340497,1.202338933944702,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,54,0.0101148645207285,1.1839776039123535,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,55,0.0099845978431403,1.193873643875122,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,56,0.010223322212696,1.1806294918060305,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,57,0.0100994843617081,1.2139291763305664,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,58,0.0105719758942723,1.2012372016906738,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,59,0.0100615346059203,1.2079083919525146,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,60,0.010135191604495,1.235825777053833,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,61,0.0102119699865579,1.2124438285827637,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,62,0.0096437899954617,1.22715425491333,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,63,0.0098920957371592,1.2255582809448242,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,64,0.00967702165246,1.202146291732788,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,65,0.0098778909817337,1.1956756114959717,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,66,0.0095820354670286,1.1980273723602295,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,67,0.0094854080677032,1.2235333919525146,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,68,0.0097034822776913,1.2775866985321045,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,69,0.0094971917942166,1.2144286632537842,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,70,0.0097823784127831,1.2294328212738037,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,71,0.0093370359204709,1.232988357543945,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,72,0.0099364022910594,1.2178897857666016,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,73,0.0099347301013767,1.2225613594055176,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,74,0.009651411473751,1.200179100036621,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,75,0.0098069887608289,1.208729267120361,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,76,0.0095253536477684,1.2664401531219482,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,77,0.0092753983847796,1.2405290603637695,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,78,0.0095426348224282,1.2393052577972412,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,79,0.0097906078211963,1.2101655006408691,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,80,0.0089718478918075,1.2226507663726809,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,81,0.0092980303056538,1.181781530380249,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,82,0.0093807338364422,1.2151193618774414,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,83,0.0092273207567632,1.215543508529663,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,84,0.0092077375017106,1.2311131954193115,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,85,0.0091077439859509,1.198394536972046,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,86,0.009335909653455,1.2004444599151611,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,87,0.0089708488434553,1.2117547988891602,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,88,0.0088016919977962,1.2288389205932615,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,89,0.0091350200772285,1.2199954986572266,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,90,0.0086712827160954,1.2171049118041992,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,91,0.0088455283269286,1.22605562210083,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,92,0.0085537268035113,1.2499427795410156,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,93,0.0087671764567494,1.2152533531188965,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,94,0.0093939660303294,1.21004056930542,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,95,0.0091199978440999,1.2201831340789795,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,96,0.008612548559904,1.2233779430389404,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,97,0.0084351436793804,1.234957218170166,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,98,0.0089494050666689,1.2466316223144531,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,99,0.0083993045613169,1.2182974815368652,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,100,0.0082656345702707,1.265721321105957,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,101,0.0083605362661182,1.2212870121002195,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,102,0.008234478700906,1.2352180480957031,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,103,0.0080597459338605,1.2646470069885254,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,104,0.0086066464148461,1.258357286453247,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,105,0.0090842749737203,1.2279603481292725,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,106,0.0083652472123503,1.2254409790039062,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,107,0.0077331262268126,1.2341911792755127,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,108,0.0078977231681346,1.2394704818725586,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,109,0.0081681085005402,1.1958611011505127,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,110,0.0082078690454363,1.21309494972229,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,111,0.0082403987646102,1.2129085063934326,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,112,0.008422610387206,1.2091238498687744,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,113,0.0082764059118926,1.2101507186889648,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,114,0.0079792981967329,1.1911327838897705,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,115,0.0078531882353127,1.1975610256195068,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,116,0.0080702902376651,1.24242901802063,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,117,0.0077734251320362,1.2454965114593506,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,118,0.0078321340493857,1.2098147869110107,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,119,0.0080087900534272,1.2035853862762451,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,120,0.0078489564172923,1.2247233390808103,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,121,0.0079426736757159,1.2151553630828855,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,122,0.0080088548362255,1.2258989810943604,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,123,0.0074296804703772,1.192399024963379,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,124,0.0075271184928715,1.241441249847412,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,125,0.0079157093726098,1.2467095851898191,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,126,0.0079573708213865,1.1966354846954346,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,127,0.0079100210405886,1.2142715454101562,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,128,0.0078801907226443,1.2117669582366943,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,129,0.0073149943165481,1.2179481983184814,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,130,0.0077527513541281,1.24829363822937,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,131,0.0074720479734241,1.238497972488403,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,132,0.0075264132395386,1.2264995574951172,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,133,0.0072967853955924,1.2387316226959229,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,134,0.0080337002500891,1.224478006362915,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,135,0.0073753671534359,1.2057433128356934,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,136,0.0074126409180462,1.2141013145446775,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,137,0.0078350370563566,1.2083683013916016,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,138,0.0078540180437266,1.204085111618042,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,139,0.0075498390942811,1.2270495891571045,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,140,0.0074423119798302,1.2169277667999268,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,141,0.0074838456325232,1.4333491325378418,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,142,0.0073352197557687,1.3525092601776123,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,143,0.0072546572051942,1.3572535514831543,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,144,0.0072644172050058,1.429680347442627,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,145,0.0072708419896662,1.460222244262695,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,146,0.0071191255934536,1.2692103385925293,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,147,0.0076076528988778,1.180653095245361,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,148,0.0075664466246962,1.2426855564117432,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,149,0.0074135676957666,1.176621675491333,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,150,0.0070817358419299,1.213360071182251,800,800,128,32,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06,0.1666666666666666,16,168116335,167895051\r\n'}
epochs = pd.concat([pd.read_csv(io.StringIO(txt)).assign(run=name) for name, txt in EPOCH_CSVS.items()], ignore_index=True)
# Plot whichever recorded loss column each run provides.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, frame in epochs.groupby("run"):
    col = "train_mse" if "train_mse" in frame else "train_loss" if "train_loss" in frame else "train_loss_mse_255"
    if col in frame: axes[0].plot(frame["epoch"], pd.to_numeric(frame[col], errors="coerce"), label=name)
    if "epoch_seconds" in frame: axes[1].plot(frame["epoch"], pd.to_numeric(frame["epoch_seconds"], errors="coerce"), label=name)
axes[0].set_title("Recorded training loss"); axes[1].set_title("Seconds per epoch")
for ax in axes: ax.set_xlabel("Epoch"); ax.grid(alpha=.25); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()
display(Markdown("ADJSCC also completed 150 epochs per channel. Its epoch CSVs remain in the linked ADJSCC Kaggle run; the aggregate is included above. The local package did not include those per-epoch rows when this notebook was assembled."))


In [ ]:
# Per-image Kodak data included in the local run artifacts.
PER_IMAGE_CSVS = {'DeepJSCC': 'model,channel,snr_db,image,psnr_db,ssim,msssim,real_symbols,complex_channel_uses,cbr\nDeepJSCC,AWGN,10.0,kodim01,21.43340263699568,0.4605445861816406,0.8283419609069824,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim02,23.698968239040124,0.5910143256187439,0.8343927264213562,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim03,24.006610609358603,0.7504670023918152,0.8772806525230408,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim04,23.17017546522488,0.6792846918106079,0.8674923181533813,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim05,20.611312598525608,0.5202287435531616,0.847913920879364,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim06,22.610474433856425,0.5678064227104187,0.8470005989074707,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim07,24.95639874355151,0.7681567072868347,0.9057396650314331,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim08,19.094292244687423,0.5025618076324463,0.8593036532402039,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim09,25.255051270718774,0.7483531832695007,0.8996027708053589,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim10,25.576342639701007,0.7348123788833618,0.8993462920188904,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim11,23.720570944911422,0.6219055652618408,0.8728143572807312,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim12,24.243731909690155,0.7341151237487793,0.8897705078125,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim13,19.882630565919644,0.4077429175376892,0.8154768943786621,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim14,22.651857278064874,0.5776296854019165,0.8569671511650085,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim15,23.569709371209644,0.7240301370620728,0.9037925004959106,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim16,26.50940003635911,0.6604882478713989,0.8779044151306152,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim17,26.42234576410829,0.751697301864624,0.9265649318695068,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim18,22.87966554918303,0.5772435665130615,0.8566379547119141,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim19,23.170345001186558,0.6355150938034058,0.8674356341362,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim20,24.810330186369953,0.778167724609375,0.9320201873779297,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim21,23.614634585756782,0.6710586547851562,0.8848098516464233,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim22,24.05299357036619,0.6279572248458862,0.849965512752533,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim23,22.706884777637597,0.7915891408920288,0.8722074031829834,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim24,22.19121671155452,0.5996863842010498,0.8735150694847107,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim01,19.860650619890706,0.3149241805076599,0.6802804470062256,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim02,19.508728632289046,0.5040397047996521,0.771343469619751,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim03,20.612716152521692,0.6964635252952576,0.776839017868042,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim04,20.202432088537805,0.6260297894477844,0.7972409129142761,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim05,18.181987429131688,0.35064029693603516,0.6647676825523376,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim06,19.820931482088678,0.46234267950057983,0.682163417339325,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim07,21.91190658404831,0.6134834289550781,0.8023084402084351,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim08,16.524364481440738,0.3449789881706238,0.690968930721283,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim09,20.732436018007245,0.6797795295715332,0.8400163650512695,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim10,23.00459230971914,0.6538388133049011,0.7631387710571289,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim11,21.733379818291905,0.5265095233917236,0.7374655604362488,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim12,20.402358157128553,0.6494988203048706,0.7035094499588013,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim13,18.256312059064484,0.2705322504043579,0.6281886100769043,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim14,20.368171323170834,0.4418865144252777,0.7307552099227905,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim15,19.597193728113695,0.5985109210014343,0.8388852477073669,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim16,20.66082663783775,0.5339365005493164,0.5478882789611816,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim17,22.63452901402316,0.505901575088501,0.8134534358978271,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim18,20.231054653580124,0.4334275424480438,0.6277710795402527,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim19,21.29343911354949,0.5581701993942261,0.798837423324585,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim20,21.606598217276407,0.7075926661491394,0.8772899508476257,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim21,20.149226063352895,0.5768390893936157,0.79123854637146,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim22,20.753081270926277,0.5374479293823242,0.7205602526664734,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim23,20.255840262506595,0.7095434069633484,0.8207847476005554,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim24,19.66194216272957,0.48059624433517456,0.764950156211853,393216,196608,0.16666666666666666\n', 'NTSCC_plus_AWGN': 'model,channel,snr_db,image,psnr_db,ssim,msssim,complex_channel_uses,cbr\nNTSCC_plus,AWGN,10.0,kodim01,21.402482986450195,0.46129482984542847,0.8203120231628418,80698.0,0.0684085422092014\nNTSCC_plus,AWGN,10.0,kodim02,20.54510498046875,0.49493059515953064,0.7243842482566833,45107.0,0.038237677680121526\nNTSCC_plus,AWGN,10.0,kodim03,17.868877410888672,0.4526536166667938,0.6827144622802734,36578.0,0.031007554796006944\nNTSCC_plus,AWGN,10.0,kodim04,20.018497467041016,0.5439622402191162,0.7269507646560669,43981.0,0.0372831556532118\nNTSCC_plus,AWGN,10.0,kodim05,20.88838768005371,0.5199162364006042,0.8467792272567749,87664.0,0.0743136935763889\nNTSCC_plus,AWGN,10.0,kodim06,22.471277236938477,0.5296902656555176,0.7983217239379883,60695.0,0.051451789008246526\nNTSCC_plus,AWGN,10.0,kodim07,23.2613525390625,0.6952646970748901,0.8366225957870483,52923.0,0.044863382975260414\nNTSCC_plus,AWGN,10.0,kodim08,19.492996215820312,0.5132796764373779,0.8578262329101562,91673.0,0.07771216498480903\nNTSCC_plus,AWGN,10.0,kodim09,22.067066192626953,0.679559588432312,0.8323910236358643,42971.0,0.03642696804470486\nNTSCC_plus,AWGN,10.0,kodim10,21.85690689086914,0.6147552728652954,0.7442799806594849,41717.0,0.0353639390733507\nNTSCC_plus,AWGN,10.0,kodim11,23.160888671875,0.5390491485595703,0.8253681659698486,55070.0,0.046683417426215276\nNTSCC_plus,AWGN,10.0,kodim12,22.814178466796875,0.6597297191619873,0.7922646403312683,36815.0,0.031208462185329862\nNTSCC_plus,AWGN,10.0,kodim13,20.12094497680664,0.3981275260448456,0.8059553503990173,84061.0,0.07125939263237847\nNTSCC_plus,AWGN,10.0,kodim14,22.51336097717285,0.5159306526184082,0.8278703689575195,68223.0,0.057833353678385414\nNTSCC_plus,AWGN,10.0,kodim15,19.49456214904785,0.5151889324188232,0.7880266904830933,44266.0,0.03752475314670139\nNTSCC_plus,AWGN,10.0,kodim16,22.62181282043457,0.5255637168884277,0.7536894083023071,43274.0,0.03668382432725695\nNTSCC_plus,AWGN,10.0,kodim17,22.66864013671875,0.5064125657081604,0.8299896121025085,45634.0,0.03868442111545139\nNTSCC_plus,AWGN,10.0,kodim18,20.902673721313477,0.49134212732315063,0.7767530679702759,65343.0,0.055391947428385414\nNTSCC_plus,AWGN,10.0,kodim19,19.633699417114258,0.4710562229156494,0.7080540657043457,52566.0,0.044560750325520836\nNTSCC_plus,AWGN,10.0,kodim20,19.591562271118164,0.6674091219902039,0.7986965179443359,41471.0,0.035155402289496526\nNTSCC_plus,AWGN,10.0,kodim21,20.8457088470459,0.5996946692466736,0.804611325263977,52223.0,0.04426998562282986\nNTSCC_plus,AWGN,10.0,kodim22,23.330095291137695,0.5710868835449219,0.7982997298240662,52982.0,0.044913397894965276\nNTSCC_plus,AWGN,10.0,kodim23,18.109561920166016,0.5803844332695007,0.749796986579895,38635.0,0.0327512953016493\nNTSCC_plus,AWGN,10.0,kodim24,21.363323211669922,0.5445016622543335,0.8232321739196777,66046.0,0.05598788791232639\n', 'NTSCC_plus_Rayleigh': 'model,channel,snr_db,image,psnr_db,ssim,msssim,complex_channel_uses,cbr\nNTSCC_plus,Rayleigh,10.0,kodim01,21.58173370361328,0.47184568643569946,0.8236074447631836,84102.0,0.07129414876302083\nNTSCC_plus,Rayleigh,10.0,kodim02,19.7816104888916,0.60387122631073,0.7401499152183533,44911.0,0.038071526421440974\nNTSCC_plus,Rayleigh,10.0,kodim03,17.923892974853516,0.5500379800796509,0.7220334410667419,37568.0,0.03184678819444445\nNTSCC_plus,Rayleigh,10.0,kodim04,22.466270446777344,0.6215094923973083,0.777235746383667,45461.0,0.0385377671983507\nNTSCC_plus,Rayleigh,10.0,kodim05,20.554140090942383,0.5000437498092651,0.8260678052902222,90529.0,0.0767423841688368\nNTSCC_plus,Rayleigh,10.0,kodim06,21.725475311279297,0.52309650182724,0.7716394662857056,63514.0,0.053841484917534724\nNTSCC_plus,Rayleigh,10.0,kodim07,24.273149490356445,0.7163235545158386,0.8531791567802429,55608.0,0.047139485677083336\nNTSCC_plus,Rayleigh,10.0,kodim08,19.416378021240234,0.5127749443054199,0.859288215637207,95472.0,0.0809326171875\nNTSCC_plus,Rayleigh,10.0,kodim09,23.22933578491211,0.6927449703216553,0.8550904989242554,44007.0,0.037305196126302086\nNTSCC_plus,Rayleigh,10.0,kodim10,22.479373931884766,0.6531922221183777,0.7774786949157715,43989.0,0.037289937337239586\nNTSCC_plus,Rayleigh,10.0,kodim11,23.435213088989258,0.5615600943565369,0.8362829685211182,57372.0,0.048634847005208336\nNTSCC_plus,Rayleigh,10.0,kodim12,22.83748435974121,0.6767486929893494,0.8081083297729492,38372.0,0.03252834743923611\nNTSCC_plus,Rayleigh,10.0,kodim13,20.0502986907959,0.3959733843803406,0.8032419681549072,87413.0,0.07410091824001735\nNTSCC_plus,Rayleigh,10.0,kodim14,22.673999786376953,0.5189183950424194,0.8265168070793152,71043.0,0.060223897298177086\nNTSCC_plus,Rayleigh,10.0,kodim15,18.024412155151367,0.5593299865722656,0.79038405418396,45178.0,0.03829786512586805\nNTSCC_plus,Rayleigh,10.0,kodim16,23.033710479736328,0.576822817325592,0.7975528240203857,45173.0,0.0382936265733507\nNTSCC_plus,Rayleigh,10.0,kodim17,22.768169403076172,0.5318491458892822,0.8255904912948608,48149.0,0.040816413031684026\nNTSCC_plus,Rayleigh,10.0,kodim18,21.339252471923828,0.49881577491760254,0.7868871688842773,67675.0,0.05736880832248264\nNTSCC_plus,Rayleigh,10.0,kodim19,19.96728515625,0.5481588244438171,0.773443341255188,54749.0,0.0464113023546007\nNTSCC_plus,Rayleigh,10.0,kodim20,14.534048080444336,0.656623363494873,0.7540795803070068,42977.0,0.0364320543077257\nNTSCC_plus,Rayleigh,10.0,kodim21,19.493276596069336,0.556827187538147,0.6827530264854431,54227.0,0.0459687974717882\nNTSCC_plus,Rayleigh,10.0,kodim22,23.88559341430664,0.5845715403556824,0.8060671091079712,55542.0,0.047083536783854164\nNTSCC_plus,Rayleigh,10.0,kodim23,17.699512481689453,0.6184325218200684,0.7684034109115601,38599.0,0.0327207777235243\nNTSCC_plus,Rayleigh,10.0,kodim24,20.903820037841797,0.5514144897460938,0.8298238515853882,68906.0,0.05841233995225695\n', 'SwinJSCC_AWGN': 'model,channel,snr_db,image,height,width,psnr_db,ssim,msssim,complex_channel_uses,cbr,inference_flops_per_image\nSwinJSCC,AWGN,10,kodim01,512,768,26.682628089473983,0.7905260324478149,0.9521769285202026,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim02,512,768,30.34579995133996,0.8080742359161377,0.9386147260665894,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim03,512,768,31.433709352789947,0.8659529685974121,0.9611327052116394,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim04,768,512,30.435943823080372,0.8407573103904724,0.9540179967880249,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim05,512,768,25.556618484730453,0.8087612986564636,0.95784592628479,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim06,512,768,27.86800833989468,0.8212150931358337,0.9518858790397644,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim07,512,768,30.71689833471642,0.8776806592941284,0.9647194147109985,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim08,512,768,24.28020568206412,0.8066482543945312,0.9564277529716492,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim09,768,512,31.508714313521892,0.8718270659446716,0.9635003805160522,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim10,768,512,30.879387804220706,0.8698203563690186,0.9644184112548828,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim11,512,768,28.936539499679206,0.8255627155303955,0.9570039510726929,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim12,512,768,31.738418347823636,0.8619880676269531,0.9588910341262817,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim13,512,768,23.855242258630387,0.7426662445068359,0.9450978636741638,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim14,512,768,27.607203809842797,0.8129513263702393,0.9553781747817993,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim15,512,768,29.73447685900848,0.8356209993362427,0.9564101099967957,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim16,512,768,30.88696893445863,0.8509318828582764,0.9618932604789734,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim17,768,512,30.705173694503422,0.8594187498092651,0.968469500541687,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim18,768,512,27.34668395526899,0.8146113753318787,0.953647792339325,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim19,768,512,29.026083399533782,0.8357560038566589,0.9567224979400635,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim20,512,768,30.294102730347745,0.864553689956665,0.9667980074882507,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim21,512,768,28.62551940264136,0.8471695780754089,0.9600932002067566,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim22,512,768,29.47523534551943,0.8258662223815918,0.9513779878616333,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim23,512,768,31.35298166017951,0.8705512881278992,0.9586695432662964,73728,0.0625,68925788161\nSwinJSCC,AWGN,10,kodim24,512,768,26.283066155175977,0.8239378929138184,0.9585350751876831,73728,0.0625,68925788161\n', 'SwinJSCC_Rayleigh': 'model,channel,snr_db,image,height,width,psnr_db,ssim,msssim,complex_channel_uses,cbr,inference_flops_per_image\nSwinJSCC,Rayleigh,10,kodim01,512,768,24.67147405615257,0.6946691870689392,0.9173280596733093,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim02,512,768,28.145205689530375,0.7469890117645264,0.9091588854789734,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim03,512,768,29.50517697262134,0.8253498077392578,0.9446656703948975,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim04,768,512,28.27218135007756,0.7779370546340942,0.9260810613632202,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim05,512,768,23.291177691722172,0.7059103846549988,0.9205960035324097,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim06,512,768,25.734887265286446,0.739762544631958,0.9202842116355896,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim07,512,768,28.18593849821911,0.8365929126739502,0.9476722478866577,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim08,512,768,22.04765469251957,0.7112292647361755,0.9266409873962402,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim09,768,512,28.98893392465903,0.8338534832000732,0.9508047103881836,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim10,768,512,28.65851807309173,0.820278525352478,0.9444183111190796,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim11,512,768,26.837850056483603,0.7479261159896851,0.9288667440414429,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim12,512,768,29.31491136342344,0.8101247549057007,0.9385780096054077,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim13,512,768,22.10798979975152,0.6196491122245789,0.8998338580131531,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim14,512,768,25.3851790783288,0.7174088954925537,0.9187666177749634,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim15,512,768,27.892427442383205,0.7932309508323669,0.9398342370986938,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim16,512,768,28.876798046837884,0.781657338142395,0.9364743232727051,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim17,768,512,28.687090605452482,0.8114039301872253,0.9475972056388855,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim18,768,512,25.355974448740337,0.7239184379577637,0.9163957834243774,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim19,768,512,26.58568295409487,0.7727519869804382,0.9318140745162964,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim20,512,768,28.523042840894544,0.8310267329216003,0.9566212892532349,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim21,512,768,26.24782736493396,0.7894060611724854,0.9419146776199341,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim22,512,768,27.354671746589357,0.7487515807151794,0.9168047308921814,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim23,512,768,28.975437789225985,0.8404704928398132,0.941547155380249,73728,0.0625,68925788161\nSwinJSCC,Rayleigh,10,kodim24,512,768,24.385031053583326,0.7429741621017456,0.9284465312957764,73728,0.0625,68925788161\n', 'DeepJSCC-Q_AWGN': 'model,channel,snr_db,image,height,width,psnr_db,ssim,msssim,complex_channel_uses,cbr,source_bits_per_symbol,training_flops_per_sample,inference_flops_per_image_128,source_commit\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim01,512,768,21.640272482643944,0.4726776480674743,0.8212048411369324,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim02,512,768,23.332191565842727,0.6355621814727783,0.8037006258964539,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim03,512,768,25.664159486772583,0.7328826785087585,0.8582655191421509,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim04,768,512,24.951217749679447,0.6664074659347534,0.8288735151290894,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim05,512,768,20.90982397139281,0.531876802444458,0.8477619886398315,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim06,512,768,22.73617098938552,0.5539206862449646,0.8210055232048035,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim07,512,768,25.16831034837397,0.7598381042480469,0.8918570876121521,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim08,512,768,19.229987009215755,0.5091602206230164,0.8495657444000244,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim09,768,512,24.14951737365167,0.7316334247589111,0.8658463358879089,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim10,768,512,24.745476694053178,0.7209782600402832,0.8661548495292664,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim11,512,768,23.85316243968393,0.6065302491188049,0.8503376841545105,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim12,512,768,23.158076205494773,0.7003903388977051,0.8393718600273132,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim13,512,768,20.08006067407079,0.4162059724330902,0.8097642660140991,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim14,512,768,23.19647669556708,0.5735020041465759,0.8485223650932312,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim15,512,768,23.91669799648279,0.7128046751022339,0.8708832263946533,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim16,512,768,25.82817659027753,0.640739381313324,0.8472427129745483,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim17,768,512,26.070410492321617,0.7261692881584167,0.8983407020568848,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim18,768,512,23.18450360215568,0.5761657357215881,0.8406535983085632,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim19,768,512,22.693618316444816,0.6241860389709473,0.8359898328781128,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim20,512,768,24.604487603395697,0.7640292644500732,0.9119941592216492,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim21,512,768,23.55006144017512,0.6505159139633179,0.8690868616104126,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim22,512,768,24.30227138418015,0.6165264844894409,0.8311516642570496,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim23,512,768,24.995803618175135,0.7617161273956299,0.8563858866691589,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),AWGN,10.0,kodim24,512,768,22.28874014069723,0.5922989249229431,0.853900671005249,196608,0.1666666666666666,4.0,168034406,167878658,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\n', 'DeepJSCC-Q_Rayleigh': 'model,channel,snr_db,image,height,width,psnr_db,ssim,msssim,complex_channel_uses,cbr,source_bits_per_symbol,training_flops_per_sample,inference_flops_per_image_128,source_commit\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim01,512,768,21.591300871518232,0.4573895335197449,0.8194543123245239,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim02,512,768,24.641124019966213,0.5890225172042847,0.7450825572013855,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim03,512,768,25.9338875018933,0.7271632552146912,0.8627669215202332,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim04,768,512,25.12115008225492,0.6789171695709229,0.8671693205833435,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim05,512,768,20.620421422191875,0.5039904117584229,0.8290707468986511,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim06,512,768,22.999143387283038,0.5486878752708435,0.8304678201675415,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim07,512,768,25.044536800622407,0.7361971735954285,0.8794648051261902,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim08,512,768,18.6555147995018,0.4806630313396454,0.8212463855743408,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim09,768,512,23.116208837600077,0.5992296934127808,0.7142400741577148,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim10,768,512,21.78802339679478,0.4852681159973144,0.6054257154464722,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim11,512,768,24.16638670348469,0.6064240336418152,0.8616046905517578,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim12,512,768,25.09599054385312,0.6989618539810181,0.8621631860733032,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim13,512,768,19.96055763240991,0.394295334815979,0.7934707403182983,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim14,512,768,23.16006521997648,0.5725263357162476,0.8566956520080566,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim15,512,768,24.441298287273383,0.710498034954071,0.8728070259094238,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim16,512,768,26.318937618632955,0.6491073369979858,0.8698251247406006,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim17,768,512,26.43856950219963,0.7424229383468628,0.9212114214897156,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim18,768,512,23.01965090329798,0.5651159286499023,0.839333713054657,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim19,768,512,22.87300982057224,0.5987911224365234,0.818671464920044,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim20,512,768,24.469277938203792,0.7515323162078857,0.9032676219940186,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim21,512,768,23.245852555721747,0.6600887775421143,0.8802236914634705,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim22,512,768,21.116195375954632,0.4160854518413543,0.5764672756195068,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim23,512,768,14.008300336398086,0.1455300450325012,0.2289957702159881,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\nDeepJSCC-Q (paper-based),Rayleigh,10.0,kodim24,512,768,21.996551873864284,0.5717597007751465,0.8407835960388184,196608,0.1666666666666666,4.0,168116335,167895051,2665e0dc6d8bf216daf9442c5d6e5d69c5ad2f06\r\n'}
per_image = pd.concat([pd.read_csv(io.StringIO(txt)).assign(run=name) for name, txt in PER_IMAGE_CSVS.items()], ignore_index=True)
display(Markdown("## Per-image Kodak records available in this notebook"))
display(per_image.head(12))
print("Embedded Kodak per-image records:", len(per_image), "(DeepJSCC and NTSCC_plus). ADJSCC per-image CSVs are linked from its Kaggle output page.")


In [ ]:
# Export reproducible report artifacts into the Kaggle run output directory.
from pathlib import Path
work = Path("/kaggle/working/consolidated_jscc_results"); work.mkdir(parents=True, exist_ok=True)
summary.to_csv(work/"five_model_kodak_summary.csv", index=False)
detail.to_csv(work/"five_model_complete_comparison.csv", index=False)
epochs.to_csv(work/"available_epoch_histories.csv", index=False)
per_image.to_csv(work/"available_per_image_kodak.csv", index=False)
print("Wrote report tables to", work)
print("Training runs (private Kaggle kernels):")
for u in [
 "https://www.kaggle.com/code/hmmkk20/deepjscc-div2k-kodak-150ep",
 "https://www.kaggle.com/code/hmmkk20/adjscc-div2k-kodak-150ep-20261016",
 "https://www.kaggle.com/code/hmmkk20/ntscc-plus-div2k-kodak-150ep-20261016",
 "https://www.kaggle.com/code/hmmkk20/swinjscc-div2k-kodak-150ep-20261018",
 "https://www.kaggle.com/code/hmmkk20/deepjscc-q-div2k-kodak-150ep-20261018",
]: print(u)
